In [ ]:
# 0) INSTALL DEPENDENCIES ───────────────────────────────────────────
!pip -q install "dash==2.16.1" "plotly==5.24.0" pandas

# 1) IMPORTS ─────────────────────────────────────────────────────────
import pandas as pd, plotly.express as px, plotly.graph_objects as go
import numpy as np, io, base64, math, re
from collections import Counter
from datetime import datetime
from plotly.colors import sample_colorscale, sequential, diverging
import warnings
import math


# ✅ Use Dash's built-in Jupyter integration (NOT the old jupyter-dash package)
from dash import Dash, dcc, html, Input, Output, State, dash_table
import dash  # keep for dash.no_update etc.


warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    module="jupyter_client.session"
)


# 2) READ INPUT (via upload) ────────────────────────────────────────
# Globals for league refs (initialized empty; updated after upload)
LG_BY_COUNT = {}
LG_PCTL     = {}
LG_RE_BY_COUNT = {}          # NEW: league run expectancy by count (RE_before)
LG_RE_TERM      = {"K": None, "BB": None}   # NEW: league RE_after for terminals

# ────────────────────────────────────────────────────────────────────
# 3) HELPERS: loader + int-count parser + run-value logic
# ────────────────────────────────────────────────────────────────────

# helper constants
STRIKE_CT    = {"0-2", "1-2", "2-2", "3-2"}
STRIKE_PITCH = {"Swinging Strike", "Called Strike",
                "Foul Tip", "Swinging Strike (Blocked)"}
FOUL_SET     = {"Foul", "Foul Tip", "Swinging Strike (Blocked)"}

# sets used for run/out classification
_RUNS = {"home run","inside the park home run","grand slam"}
_OUTS = {"groundout","flyout","lineout","pop out","double play","triple play",
         "fielders choice out","sac fly","sac bunt","foul out","forceout",
         "strikeout","strikeout - dp"}
_NO_OUT = {"single","double","triple","home run","error","fielder's choice",
           "fielders choice", "catcher's interference", "hit by pitch",
           "bunt single"}

DT_CELL = {
    "padding": "4px 6px",
    "fontFamily": "Arial",
    "fontSize": "13px",
    "textAlign": "right",
    "whiteSpace": "normal",
    "height": "auto",
    "color": "#000",
}
DT_HEAD = {"fontWeight": "bold", "textAlign": "center", "backgroundColor": "#f4f6f6"}
DT_TABLE = {
    "width": "100%",
    "overflowX": "visible",   # no horizontal scroll in the left rail
    "border": "none",
}

RATE_WIDTHS = [
    {"if": {"column_id": "PitchType"}, "textAlign": "center", "width": "58px"},
    {"if": {"column_id": "SwingPct"},  "width": "70px"},
    {"if": {"column_id": "WhiffPct"},  "width": "70px"},
    {"if": {"column_id": "CalledPct"}, "width": "74px"},
    {"if": {"column_id": "CSW_Pct"},   "width": "72px"},
    {"if": {"column_id": "StrikePct"}, "width": "74px"},
    {"if": {"column_id": "Pitches"},   "width": "64px"},
]

# STRIKE ZONE (feet)
X_MIN, X_MAX = -0.83, 0.83
Z_BOT, Z_TOP = 1.5, 3.5
Z_YCTR = (Z_BOT + Z_TOP) / 2.0

FASTBALLS = {"FF","FA","SI","FT","FC","CT"}
BREAKERS  = {"SL","CU","KC","KN","SV"}     # add/remove as your feed uses
OFFSPEED  = {"CH","FS","SF","FO","SC"}     # "SC" sometimes = screw/change in some feeds

PITCHCLASS_COLOR_MAP = {
    "Fastball": "#d62728",      # red
    "Breaker":  "#1f77b4",      # blue
    "Offspeed": "#2ca02c",      # green
    "Other":    "#7f7f7f",      # grey
    "Unclassified": "#bdbdbd",  # light grey
}

COUNTGROUP_COLOR_MAP = {
    "Hitter":  "#d62728",  # red
    "Even":    "#7f7f7f",  # grey
    "Pitcher": "#2ca02c",  # green
}


def add_rate_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds unified per-pitch flags used across the app:
      is_swing, is_whiff, is_called, is_csw, is_strike
    """
    if df.empty:
        # still return the same columns for callers that expect them
        for c in ["is_swing","is_whiff","is_called","is_csw","is_strike"]:
            df[c] = []
        return df

    out = df.copy()
    out["is_swing"]  = out["PitchResult"].isin(swing_results).astype(int)
    out["is_whiff"]  = out["PitchResult"].isin(WHIFF_SET).astype(int)
    out["is_called"] = out["PitchResult"].eq("Called Strike").astype(int)
    out["is_csw"]    = out["PitchResult"].isin(CSW_SET).astype(int)
    out["is_strike"] = out["PitchResult"].isin(ALL_STRIKE_SET).astype(int)
    return out


def count_class(node: str) -> str:
    """
    Classify node for color scale selection.
    K  -> pitcher class, BB -> hitter class
    """
    if node == "K":   return "pitcher"
    if node == "BB":  return "hitter"
    if node in PITCHER_CT: return "pitcher"
    if node in HITTER_CT:  return "hitter"
    if node in EVEN_CT:    return "even"
    # Any other display nodes (e.g., 3-2) default neutral grey ramp
    return "even"

def _pitch_class(series: pd.Series) -> pd.Series:
    s = series.astype("string")
    s = s.str.upper().str.strip()

    out = pd.Series(np.full(len(s), "Other", dtype=object), index=s.index)
    out[s.isna()] = "Unclassified"
    out[s.isin(list(FASTBALLS))] = "Fastball"
    out[s.isin(list(BREAKERS))]  = "Breaker"
    out[s.isin(list(OFFSPEED))]  = "Offspeed"
    return out

def _count_group(df: pd.DataFrame) -> pd.Series:
    # Prefer numeric balls/strikes if present, else parse Count like "1-2"
    if "Balls_int" in df.columns and "Strikes_int" in df.columns:
        b = pd.to_numeric(df["Balls_int"], errors="coerce")
        k = pd.to_numeric(df["Strikes_int"], errors="coerce")
    elif "Balls_pre" in df.columns and "Strikes_pre" in df.columns:
        b = pd.to_numeric(df["Balls_pre"], errors="coerce")
        k = pd.to_numeric(df["Strikes_pre"], errors="coerce")
    else:
        # parse Count "B-S"
        parts = df.get("Count", pd.Series([pd.NA]*len(df))).astype("string").str.split("-", expand=True)
        b = pd.to_numeric(parts[0], errors="coerce") if parts.shape[1] > 0 else pd.Series([np.nan]*len(df))
        k = pd.to_numeric(parts[1], errors="coerce") if parts.shape[1] > 1 else pd.Series([np.nan]*len(df))

    out = pd.Series(np.full(len(df), "Even", dtype=object), index=df.index)
    out[(b > k)] = "Hitter"
    out[(b < k)] = "Pitcher"
    # if b or k missing, keep "Even" (neutral) so it won't blow up
    return out


# ── Pitch abbreviations (display-compact) ─────────────────────────
PITCH_ABBR = {
    "FOUR-SEAM FASTBALL": "FF",
    "FOUR SEAM FASTBALL": "FF",
    "4-SEAM FASTBALL":    "FF",
    "4-SEAM":             "FF",
    "FASTBALL":           "FF",
    "SINKER":             "SI",
    "TWO-SEAM FASTBALL":  "SI",
    "CUTTER":             "FC",
    "SLIDER":             "SL",
    "CURVEBALL":          "CB",
    "KNUCKLE CURVE":      "KC",
    "CHANGEUP":           "CH",
    "SPLIT-FINGER":       "SF",
    "SPLITTER":           "SF",
    "SWEPPER":            "SW",
    "SWEEPER":            "SW",
    "SCREWBALL":          "SC",
    "EEPHUS":             "EP",
    "KNUCKLEBALL":        "KN",
    "SLURVE":             "SV",
}

REV_ABBR = {v:k for k,v in PITCH_ABBR.items()}


# ── THEME: unified color grammar ───────────────────────────────────

# Diverging Red→White→Green (white is the midpoint / “average”)
RDGN_WHITE_MID = [
    [0.00, "rgb(192,57,43)"],   # red   (bad for pitcher)
    [0.50, "rgb(255,255,255)"], # white (neutral / average)
    [1.00, "rgb(39,174,96)"],   # green (good for pitcher)
]

THEME = {
    # sequential ramp for “more is better” → pure Greens
    "seq":  sequential.Greens,
    # diverging ramp for above/below average → Red–White–Green
    "div":  RDGN_WHITE_MID,
    "percent_bins": [0, 20, 40, 60, 80, 100],
    "rv_clip": 0.03,
}

TILE_CLIP = THEME["rv_clip"]
MIN_PITCHTYPE_USAGE_TILE = 0.03

# ------------------------------
# PitchType standardization + 3% keep-set
# ------------------------------
_MISSING_PT_TOKENS = {"", "nan", "NaN", "NAN", "<NA>"}

def _pt_std(s: pd.Series) -> pd.Series:
    """
    Normalize PitchType safely:
      - strip
      - treat empty/"nan"/"<NA>" as NA
      - uppercase for consistent joins
    """
    s = s.astype("string").str.strip()
    s = s.where(~s.isin(_MISSING_PT_TOKENS), pd.NA)
    return s.str.upper()

def pitchtype_keep_set(df: pd.DataFrame, threshold: float = MIN_PITCHTYPE_USAGE_TILE, col: str = "PitchType") -> set:
    """
    Returns the set of PitchTypes that meet the threshold, where usage is computed
    ONLY over tagged pitch types (PitchType not NA after cleaning).
    """
    if df is None or df.empty or col not in df.columns:
        return set()

    pt = _pt_std(df[col])
    known = pt.dropna()
    denom = int(len(known))
    if denom == 0:
        return set()

    usage = known.value_counts() / denom
    return set(usage[usage >= threshold].index.tolist())

def nan_pitchtype_stats(df: pd.DataFrame, col: str = "PitchType"):
    """
    NaN PitchType count/percent (percent is vs ALL pitches, not tagged-only).
    """
    if df is None or df.empty or col not in df.columns:
        return 0, 0, 0.0

    pt = _pt_std(df[col])
    total = int(len(pt))
    nan_n = int(pt.isna().sum())
    pct = (nan_n / total * 100) if total else 0.0
    return nan_n, total, pct


# ── Extra ramps (neutral + leverage helpers) ─────────────────────
THEME.update({
    "neutral":  sequential.Greys,    # for “not value-laden” amounts
    "usage":    sequential.Blues,    # usage %
    "trans":    sequential.Purples,  # transition %
})

# Anchor node palettes at WHITE so the ramps literally start at white,
# but use multi-stop ramps with more saturation/contrast.
WHITE = "rgb(255,255,255)"
THEME.update({
    "node_pitcher": [
        [0.00, WHITE],
        [0.20, sequential.Greens[2]],   # pale green
        [0.55, sequential.Greens[6]],   # saturated mid
        [1.00, sequential.Greens[-1]],  # deep green
    ],
    "node_hitter": [
        [0.00, WHITE],
        [0.20, sequential.Reds[2]],     # pale red
        [0.55, sequential.Reds[6]],     # saturated mid
        [1.00, sequential.Reds[-1]],    # deep red
    ],
    "node_even": [
        [0.00, WHITE],
        [0.35, "rgb(160,160,160)"],     # mid grey
        [0.70, "rgb(95,95,95)"],        # dark grey
        [1.00, "rgb(30,30,30)"],        # near-black for max contrast
    ],
})


# ---------- TABLE UTILITIES: unified diverging shading + rate table ----------

def _rgba_from_rgb_str(s: str, alpha: float = 0.95) -> str:
    # s like "rgb(255,0,0)"
    r, g, b = [int(x) for x in s.strip("rgb()").split(",")]
    return f"rgba({r},{g},{b},{alpha})"

def _col_minmax(data, col):
    vals = [r.get(col) for r in data]
    vals = [float(v) for v in vals if v is not None and v == v]  # drop NAs
    if not vals:
        return 0.0, 0.0
    return min(vals), max(vals)

def per_column_diverging_shading(data, cols, *, alpha=0.85):
    """
    Continuous gradient shading per column.
    Red–white–green scale, but smooth.
    min(col) = strongest red
    max(col) = strongest green
    Everything else interpolates continuously.
    """
    styles = []
    if not data:
        return styles

    for c in cols:
        # Collect values
        vals = []
        for r in data:
            v = r.get(c)
            if v is not None and v == v:
                vals.append(float(v))

        if not vals:
            continue

        lo, hi = min(vals), max(vals)
        rng = hi - lo if hi != lo else 1.0

        for r in data:
            v = r.get(c)
            if v is None or v != v:
                continue

            # normalize 0-1
            t = (float(v) - lo) / rng
            t = float(np.clip(t, 0.0, 1.0))

            # continuous color from THEME['div']
            rgb = sample_colorscale(THEME["div"], [t])[0]  # "rgb(...)"
            r_, g_, b_ = [int(x) for x in rgb.strip("rgb()").split(",")]
            bg = f"rgba({r_},{g_},{b_},{alpha})"

            styles.append({
                "if": {"filter_query": f"{{{c}}} = {v}", "column_id": c},
                "backgroundColor": bg,
                "color": "#000",       # ALWAYS BLACK TEXT
            })

    return styles


def rowwise_ygnbu_shading(data, *, key_col, cols, alpha=0.25):
    """
    Keeps the old function name for compatibility, but uses THEME['div'].
    Shades row-wise: each row’s min→max mapped to RdYlGn.
    """
    if not data:
        return []
    styles = []
    for r in data:
        key = r.get(key_col, "")
        vals = [r.get(c) for c in cols]
        vals = [float(v) for v in vals if v is not None and v == v]
        if not vals:
            continue
        lo, hi = min(vals), max(vals)
        for c in cols:
            v = r.get(c, None)
            if v is None or v != v or hi == lo:
                idx = 0.5
            else:
                idx = float(np.clip((float(v)-lo)/(hi-lo), 0, 1))
            rgb = sample_colorscale(THEME["div"], [idx])[0]
            bg  = _rgba_from_rgb_str(rgb, alpha)
            fg  = text_color_for(bg)
            styles.append({
                "if": {"filter_query": f'{{{key_col}}} = "{key}"', "column_id": c},
                "backgroundColor": bg,
                "color": "#000"
            })
    return styles

def make_rate_table(df, count_set, *, sort_key="CSW_Pct", desc=True, use_pre=True):
    # choose the right count column
    count_col = "Count_pre" if (use_pre and "Count_pre" in df.columns) else "Count"
    sub = df[df[count_col].isin(count_set)].copy()
    if sub.empty:
        cols = [{"name": n, "id": n} for n in ["PitchType"] + RATE_COLS + ["Pitches"]]
        return [], cols

    # --- 3% overall usage threshold (computed vs tagged pitch types ONLY) ---
    keep = pitchtype_keep_set(df, MIN_PITCHTYPE_USAGE_TILE, col="PitchType")
    sub["_PT"] = _pt_std(sub["PitchType"])
    sub = sub[sub["_PT"].isin(keep)].copy()
    sub["PitchType"] = sub["_PT"]


    # 🔐 normalize *here* so string mismatches can’t zero-out CalledPct
    sub["PitchResult"] = (
        sub["PitchResult"].astype(str)
            .str.replace("\u00a0", " ", regex=False)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.title()
    )

    # flags (unchanged)
    sub["is_swing"]  = sub["PitchResult"].isin(swing_results).astype(int)
    sub["is_whiff"]  = sub["PitchResult"].isin(WHIFF_SET).astype(int)
    sub["is_called"] = sub["PitchResult"].eq("Called Strike").astype(int)
    sub["is_csw"]    = sub["PitchResult"].isin(CSW_SET).astype(int)
    sub["is_strike"] = sub["PitchResult"].isin(ALL_STRIKE_SET).astype(int)

    g = sub.groupby("PitchType")
    n = g.size().rename("Pitches")
    out = pd.DataFrame({
        "PitchType": n.index,
        "SwingPct":  (g["is_swing"].mean()*100).round(1).values,
        "WhiffPct":  (g["is_whiff"].mean()*100).round(1).values,
        "CalledPct": (g["is_called"].mean()*100).round(1).values,
        "CSW_Pct":   (g["is_csw"].mean()*100).round(1).values,
        "StrikePct": (g["is_strike"].mean()*100).round(1).values,
        "Pitches":   n.values
    }).sort_values(sort_key, ascending=not desc)

    out["PitchType"] = out["PitchType"].map(pitch_abbr)
    # keep one decimal place consistently
    for c in ["SwingPct","WhiffPct","CalledPct","CSW_Pct","StrikePct"]:
        out[c] = out[c].map(lambda x: None if pd.isna(x) else round(float(x), 1))

    cols = [{"name": c, "id": c} for c in out.columns]
    return out.to_dict("records"), cols


# Subtle "leverage" ramp: light grey → amber → deep purple
LEVERAGE_SCALE = [
    [0.00, "rgb(235,235,235)"],
    [0.35, "rgb(210,210,210)"],
    [0.60, "rgb(255,196,77)"],   # amber pop when leverage emerges
    [0.80, "rgb(200,120,200)"],
    [1.00, "rgb(120,60,160)"],
]


# === Diverging (red ➜ #f7e379 ➜ green) column-wise shading =========

def rgba_from_plotly(rgb: str, a: float) -> str:
    r, g, b = [int(x) for x in rgb.strip("rgb()").split(",")]
    return f"rgba({r},{g},{b},{a})"

def seq_shades(n=5, alpha=0.22):
    # evenly-spaced picks from the same sequential ramp used in Plinko
    ps = np.linspace(0.05, 0.95, n)
    return [rgba_from_plotly(sample_colorscale(THEME["seq"], [p])[0], alpha) for p in ps]

def div_colour(v: float, lo: float, hi: float, alpha=1.0):  # default α=1
    if lo == hi:
        idx = 0.5
    else:
        idx = (v - lo) / (hi - lo)
    idx = float(np.clip(idx, 0, 1))
    rgb = sample_colorscale(THEME["div"], [idx])[0]  # "rgb(r,g,b)"
    # force opaque
    r, g, b = [int(x) for x in rgb.strip("rgb()").split(",")]
    return f"rgba({r},{g},{b},1)"


def readable_text_on(rgb: str) -> str:
    # auto-contrast for text placed on colored fills
    r, g, b = [int(x) for x in rgb.strip("rgb()").split(",")]
    L = 0.2126*(r/255)**2.2 + 0.7152*(g/255)**2.2 + 0.0722*(b/255)**2.2
    return "#000" if L > 0.5 else "#fff"

def text_color_for(bg: str) -> str:
    # accepts "rgb(...)" or "rgba(...)"
    nums = bg.strip().split("(", 1)[1].split(")")[0].split(",")[:3]
    r, g, b = [int(float(x)) for x in nums]
    # WCAG-ish luminance
    L = 0.2126*(r/255)**2.2 + 0.7152*(g/255)**2.2 + 0.0722*(b/255)**2.2
    return "#000" if L > 0.50 else "#fff"

def good_bad_bg(score, clip=TILE_CLIP, alpha=1.0):  # was 0.95
    GREEN = (39, 174, 96)
    RED   = (192, 57, 43)
    WHITE = (255, 255, 255)

    t = float(np.clip(abs(score) / clip, 0.0, 1.0))
    t = t ** 0.6   # <<< gamma: makes small differences look stronger
    tgt = GREEN if score > 0 else RED if score < 0 else (220, 220, 220)

    r = int(WHITE[0] + (tgt[0] - WHITE[0]) * t)
    g = int(WHITE[1] + (tgt[1] - WHITE[1]) * t)
    b = int(WHITE[2] + (tgt[2] - WHITE[2]) * t)
    return f"rgba({r},{g},{b},{alpha})"


def pitch_abbr(name: str) -> str:
    if not isinstance(name, str):
        return str(name)
    key = name.strip().upper()

    # 🔒 If it already looks like an abbreviation (1–3 letters), keep it.
    # Prevents "SL" → "S", "CH" → "C", etc. on a second pass.
    if re.fullmatch(r"[A-Z]{1,3}", key):
        return key

    if key in PITCH_ABBR:
        return PITCH_ABBR[key]
    parts = [w for w in re.split(r"[\s\-]+", key) if w]
    return "".join(p[0] for p in parts)[:3]



def unify_pitch_result(row: dict[str, any]) -> str:
    pr = str(row.get("PitchResult","")).strip()
    pr_lc = pr.lower()


    # In-play: use PLAY_RESULT to bucket
    if pr_lc.startswith("in play"):
        play = str(row.get("PLAY_RESULT","")).strip().lower()
        if play in _RUNS:   return "In Play, Run(s)"
        if play in _OUTS:   return "In Play, Out(s)"
        if play in _NO_OUT: return "In Play, No Out"
        return "In Play, No Out"

    # Ball variants → Ball
    if pr_lc in ("ball in dirt", "pitchout"):
        return "Ball"

    # Preserve bunt labels
    if pr_lc in ("foul bunt", "missed bunt", "bunt foul tip"):
        return pr.title()

    # Everything else → title case ("swinging strike (blocked)" → "Swinging Strike (Blocked)")
    return pr.title()

# ======= UNIFIED TABLE HELPERS (adds Whiff%, CSW%, Strike% etc.) =======
WHIFF_SET       = {"Swinging Strike", "Swinging Strike (Blocked)"}
ALL_STRIKE_SET  = STRIKE_PITCH | {"Foul"}      # include fouls in total Strike%

RATE_COLS = ["SwingPct","WhiffPct","CalledPct","CSW_Pct","StrikePct"]


def precount_from_post(b_post, s_post, pr):
    """
    Given post-pitch (balls, strikes) and PitchResult, return (balls_pre, strikes_pre).
    Handles 2-strike fouls and K/BB terminals.
    """
    # coerce
    b = int(b_post); s = int(s_post)
    pr = str(pr).strip()

    # In-play results don't change the count (post == pre)
    if pr in {"In Play, No Out", "In Play, Out(s)", "In Play, Run(s)"}:
        return b, s

    # Balls
    if pr == "Ball":
        # If this ball *walked* the hitter, your feed will often show s==2, b==3 anyway.
        # The pre-pitch state is one fewer ball.
        return max(0, b - 1), s

    # Foul logic (remember the 2-strike freeze)
    if pr == "Foul":
        if s >= 2:
            return b, 2       # foul at 2 strikes doesn't advance strikes
        return b, max(0, s - 1)

    # All other "strike" flavors
    if pr in {"Called Strike", "Swinging Strike", "Foul Tip", "Swinging Strike (Blocked)"}:
        return b, max(0, s - 1)

    # Bunt edge-cases: treat like strikes for pre-count,
    # with the same 2-strike freeze for foul-bunt
    if pr in {"Foul Bunt", "Bunt Foul Tip"}:
        if s >= 2:
            return b, 2
        return b, max(0, s - 1)
    if pr == "Missed Bunt":
        return b, max(0, s - 1)

    # Unknowns → assume no change
    return b, s



def putaway_metrics(df):
    """
    Returns:
        kill      – % of PAs that reach two strikes and K on the *next* pitch
        pats      – avg pitches AFTER the first 2-strike pitch (PATS)
        foul_avg  – avg foul balls AFTER first 2-strike pitch per two-strike PA
        ball_avg  – avg balls AFTER first 2-strike pitch per two-strike PA
        ipno_pct  – % of two-strike pitches that are 'In Play, No Out' (per-pitch)
    """
    if df.empty:
        return dict(kill=0.0, pats=0.0, foul_avg=0.0, ball_avg=0.0, ipno_pct=0.0)

    kills = 0
    foul_cts, ball_cts, pats_list = [], [], []

    # per-pitch flag for ipno_pct later
    two_mask = df["Count"].isin({"0-2","1-2","2-2","3-2"})
    ipno_pct = df.loc[two_mask, "PitchResult"].eq("In Play, No Out").mean() * 100 if two_mask.any() else 0.0

    # walk true ABs
    for _, ab in _group_by_ab(df):
        # ensure order inside AB
        if "PITCH_NUMBER_AB" in ab.columns and ab["PITCH_NUMBER_AB"].notna().any():
            ab = ab.sort_values("PITCH_NUMBER_AB")
        else:
            ab = ab.sort_values("DateTime")

        ab = ab.reset_index(drop=True)
        two_idx = ab.index[ab["Count"].isin({"0-2","1-2","2-2","3-2"})]
        if len(two_idx) == 0:
            continue

        i0 = int(two_idx[0])        # first two-strike pitch in this PA
        after = ab.iloc[i0+1:]      # everything AFTER first two-strike pitch

        # KILL%: next pitch is a strike-type AND ends PA immediately
        if not after.empty:
            nxt = after.iloc[0]
            ends_pa_right_after = (i0 + 1 == len(ab) - 1)  # next pitch is last in PA
            if (nxt["PitchResult"] in STRIKE_PITCH) and ends_pa_right_after:
                kills += 1

        foul_cts.append(after["PitchResult"].isin(FOUL_SET).sum())
        ball_cts.append(after["PitchResult"].eq("Ball").sum())
        pats_list.append(len(after))

    total_two_strike_PAs = len(pats_list)
    kill_pct = (kills / total_two_strike_PAs * 100) if total_two_strike_PAs else 0.0
    pats     = float(np.mean(pats_list)) if pats_list else 0.0
    foul_avg = float(np.mean(foul_cts))  if foul_cts else 0.0
    ball_avg = float(np.mean(ball_cts))  if ball_cts else 0.0

    return dict(kill=kill_pct, pats=pats, foul_avg=foul_avg, ball_avg=ball_avg, ipno_pct=ipno_pct)


# ------------------------------------------------------------------
# CSW PITCH-RESULTS  (global constant – visible to every helper)
# ------------------------------------------------------------------
CSW_SET = {
    "Swinging Strike",
    "Called Strike",
    "Foul Tip",
    "Swinging Strike (Blocked)",
}

# ─── MIX / REPEAT METRICS ───────────────────────────────────────────
def mix_metrics(df, bucket_mask):
    """
    Returns plain numbers; caller formats them.
       blend_pct   – Even-Mix %  (0-100, higher = harder to sit on one pitch)
       repeat_pct  – share of back-to-back same-pitch (0-100)
    """
    seq = df.loc[bucket_mask, "PitchType"].tolist()
    n   = len(seq)
    if n < 10:
        return dict(blend_pct=np.nan, repeat_pct=np.nan)

    # how evenly are pitches distributed?
    cnts   = Counter(seq).values()
    probs  = np.array(list(cnts)) / n
    entropy= -(probs * np.log2(probs)).sum()
    e_max  = np.log2(len(cnts))
    blend_pct = (entropy / e_max) * 100           # 0-100%

    repeats    = sum(a == b for a, b in zip(seq[:-1], seq[1:]))
    repeat_pct = repeats / (n - 1) * 100

    return dict(blend_pct=blend_pct, repeat_pct=repeat_pct)


# ─── GET-ME-RIGHT  (robust, whitespace-safe) ───────────────────────
def get_me_right(df):
    """
    Prev-miss ➜ Helper ➜ Prev-CSW, grouped by *true* AB order.
    returns PrevPitch | HelperPitch | Success | Attempts | SuccessRate
    """
    if df.empty:
        return pd.DataFrame(columns=["PrevPitch","HelperPitch","Success","Attempts","SuccessRate"])

    df = df.copy()
    df["pt_std"] = df["PitchType"].astype(str).str.strip().str.upper()

    records = []
    for _, ab in _group_by_ab(df):
        ab = ab.sort_values("PITCH_NUMBER_AB") if "PITCH_NUMBER_AB" in ab.columns and ab["PITCH_NUMBER_AB"].notna().any() else ab.sort_values("DateTime")
        ab["prev1_pt"]  = ab["pt_std"].shift(1)
        ab["prev2_pt"]  = ab["pt_std"].shift(2)
        ab["prev2_res"] = ab["PitchResult"].shift(2)
        trip = ab[ab["pt_std"].eq(ab["prev2_pt"])].dropna(subset=["prev1_pt","prev2_pt"])
        if trip.empty:
            continue
        trip["key"] = list(zip(trip["prev2_pt"], trip["prev1_pt"]))
        attempts  = trip["key"].value_counts()
        successes = trip.loc[(~trip["prev2_res"].isin(CSW_SET)) & (trip["PitchResult"].isin(CSW_SET)), "key"].value_counts()
        for k, att in attempts.items():
            succ = int(successes.get(k, 0))
            records.append(dict(
                PrevPitch   = k[0],
                HelperPitch = k[1],
                Success     = succ,
                Attempts    = int(att),
                SuccessRate = succ / att if att else 0.0
            ))

    if not records:
        return pd.DataFrame(columns=["PrevPitch","HelperPitch","Success","Attempts","SuccessRate"])
    return (pd.DataFrame(records)
              .groupby(["PrevPitch","HelperPitch"], as_index=False)
              .sum()
              .assign(SuccessRate=lambda t: t["Success"] / t["Attempts"])
              .sort_values(["SuccessRate","Attempts"], ascending=[False, False]))

MIN_PITCHES_RV = 15          # 🟢 only colour counts seen ≥ 15×
RV_CLIP        = THEME["rv_clip"]


def _find_col_case_insensitive(df: pd.DataFrame, candidates: list[str]) -> str | None:
    cols = {c.lower(): c for c in df.columns}
    for name in candidates:
        c = cols.get(name.lower())
        if c:
            return c
    return None

def _count_from_balls_strikes(df: pd.DataFrame) -> None:
    """
    Canonical count for ALL count-based tables/tiles = **PRE-PITCH** count.
    We also keep post-pitch count fields for sequencing/Plinko transitions.

    Outputs:
      - Balls_pre / Strikes_pre   (count BEFORE pitch)
      - Count_pre                = "B-S" before pitch
      - Balls_post / Strikes_post (count AFTER pitch)
      - Balls_int / Strikes_int   (clipped post for display/Plinko)
      - Count_post               = "B-S" after pitch (display cap 3/2)
      - Count                    = Count_pre  ✅ canonical everywhere else
    """
    bcol = _find_col_case_insensitive(df, ["Balls","BALLS","B","ball","balls","num_balls"])
    scol = _find_col_case_insensitive(df, ["Strikes","STRIKES","S","strike","strikes","num_strikes"])

    balls_raw = pd.to_numeric(df[bcol], errors="coerce").astype("Int64") if bcol else pd.Series([pd.NA]*len(df), dtype="Int64")
    strikes_raw = pd.to_numeric(df[scol], errors="coerce").astype("Int64") if scol else pd.Series([pd.NA]*len(df), dtype="Int64")

    # Ensure PitchResult exists (needed to derive post from pre, or pre from post)
    if "PitchResult" not in df.columns:
        df["PitchResult"] = pd.NA

    # Heuristic: if first pitch of AB is usually 0-0, feed is PRE count.
    # Otherwise treat feed as POST count.
    looks_pre = True
    if "PITCH_NUMBER_AB" in df.columns and df["PITCH_NUMBER_AB"].notna().any():
        pn = pd.to_numeric(df["PITCH_NUMBER_AB"], errors="coerce")
        first_mask = pn.eq(1)
        if first_mask.any():
            frac_00 = ((balls_raw[first_mask] == 0) & (strikes_raw[first_mask] == 0)).mean()
            if pd.notna(frac_00) and frac_00 < 0.5:
                looks_pre = False

    STRIKE_PITCH = {"Called Strike", "Swinging Strike", "Swinging Strike (Blocked)"}

    def _post_from_pre(b: int, s: int, pr: str) -> tuple[int, int]:
        pr = (pr or "").strip()
        if pr == "Ball":
            return b + 1, s
        if pr == "Foul":
            return (b, s + 1) if s < 2 else (b, s)
        if pr in STRIKE_PITCH:
            return b, s + 1
        # In play / Unknown / anything else: count doesn't advance for our purposes
        return b, s

    if looks_pre:
        # Feed is PRE
        df["Balls_pre"] = balls_raw
        df["Strikes_pre"] = strikes_raw

        df["Balls_post"] = pd.Series([pd.NA]*len(df), dtype="Int64")
        df["Strikes_post"] = pd.Series([pd.NA]*len(df), dtype="Int64")

        mask = df["Balls_pre"].notna() & df["Strikes_pre"].notna()
        if mask.any():
            post = df.loc[mask, ["Balls_pre","Strikes_pre","PitchResult"]].apply(
                lambda r: _post_from_pre(int(r["Balls_pre"]), int(r["Strikes_pre"]), str(r["PitchResult"])),
                axis=1
            )
            df.loc[post.index, "Balls_post"]   = [b for b, _ in post]
            df.loc[post.index, "Strikes_post"] = [s for _, s in post]
    else:
        # Feed is POST
        df["Balls_post"] = balls_raw
        df["Strikes_post"] = strikes_raw

        df["Balls_pre"] = pd.Series([pd.NA]*len(df), dtype="Int64")
        df["Strikes_pre"] = pd.Series([pd.NA]*len(df), dtype="Int64")

        mask = df["Balls_post"].notna() & df["Strikes_post"].notna()
        if mask.any():
            pre = df.loc[mask, ["Balls_post","Strikes_post","PitchResult"]].apply(
                lambda r: precount_from_post(int(r["Balls_post"]), int(r["Strikes_post"]), str(r["PitchResult"])),
                axis=1
            )
            df.loc[pre.index, "Balls_pre"]   = [b for b, _ in pre]
            df.loc[pre.index, "Strikes_pre"] = [s for _, s in pre]

    # Clip PRE for display / stability
    df["Balls_pre"]   = df["Balls_pre"].astype("Int64").clip(0, 3)
    df["Strikes_pre"] = df["Strikes_pre"].astype("Int64").clip(0, 2)
    df["Count_pre"]   = df.apply(
        lambda r: f"{int(r['Balls_pre'])}-{int(r['Strikes_pre'])}"
        if pd.notna(r["Balls_pre"]) and pd.notna(r["Strikes_pre"]) else None,
        axis=1
    )

    # POST + clipped POST for Plinko transitions
    df["Balls_post"]   = df["Balls_post"].astype("Int64")
    df["Strikes_post"] = df["Strikes_post"].astype("Int64")
    df["Balls_int"]   = df["Balls_post"].clip(0, 3)
    df["Strikes_int"] = df["Strikes_post"].clip(0, 2)
    df["Count_post"]  = df["Balls_int"].astype(str) + "-" + df["Strikes_int"].astype(str)

    # ✅ Canonical Count everywhere else
    df["Count"] = df["Count_pre"]


def load_pitchlog(data_bytes):
    buf = io.BytesIO(data_bytes)
    df = pd.read_csv(buf, dtype=str) if b',' in data_bytes[:100] else pd.read_excel(buf)

    # hard-normalize early
    for col in ("PitchResult", "PitchType", "PLAY_RESULT"):
        if col in df.columns:
            df[col] = (
                df[col].astype("string")   # keeps <NA>, doesn't create "nan"
                      .str.replace("\u00a0", " ", regex=False)
                      .str.replace("\t", " ", regex=False)
                      .str.strip()
            )

    # make sure common fake-missing strings become real missing
    if "PitchType" in df.columns:
        df["PitchType"] = df["PitchType"].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})


    # Legacy remaps (unchanged)
    legacy_map = {
        "BREWERS_PITCHER":"Pitcher",
        "PITCH_TYPE"     :"PitchType",
        "PITCH_CALL"     :"PitchResult",
        "VELOCITY"       :"Velo",
        "COUNT"          :"RawCount",
        "OUTS"           :"OUTS",
    }
    for src, dst in legacy_map.items():
        if src in df.columns and dst not in df.columns:
            df = df.rename(columns={src: dst})

    # DateTime/GameDate (unchanged convenience)
    if "DateTime" not in df.columns:
        for cand in ["DATE_TIME","Date","Datetime","date_time"]:
            if cand in df.columns:
                df["DateTime"] = pd.to_datetime(df[cand], errors="coerce")
                break
    if "DateTime" in df.columns and "GameDate" not in df.columns:
        dt = pd.to_datetime(df["DateTime"], errors="coerce")
        df["GameDate"] = np.where(
            dt.dt.hour < 4, (dt - pd.Timedelta(days=1)).dt.date, dt.dt.date
        )

    # NEW: coerce your sequencing truth columns
    for c in ["AT_BAT_NUMBER","PITCH_NUMBER_AB","PITCH_NUMBER_GAME"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    # Unify pitch results FIRST
    if "PitchResult" in df.columns:
        df["PitchResult"] = df.apply(unify_pitch_result, axis=1)
        df["PitchResult"] = (
            df["PitchResult"].astype(str)
                            .str.replace("\u00a0", " ", regex=False)
                            .str.strip()
        )

    # THEN derive Count fields (Count becomes PRE-pitch)
    _count_from_balls_strikes(df)


    # Batter side (best-effort; unchanged)
    if "BatterHand" not in df.columns:
        for cand in ["BATTER_SIDE","BATTER_HAND"]:
            if cand in df.columns:
                df["BatterHand"] = df[cand].fillna("All").astype(str)
                break
        else:
            df["BatterHand"] = "All"

    # Numeric kinematics
    for col in ["Velo","REL_HEIGHT","REL_SIDE","EXTENSION",
                "INDUCED_VERT_BREAK","HORZ_BREAK","SPIN_RATE",
                "EXIT_VELO","LAUNCH_ANGLE","pfx_x","pfx_z",
                "plate_loc_side","plate_loc_height"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # --- Movement from pfx_* (already in inches, feed is catcher's view) ---
    def _find_ci(colname: str) -> str | None:
        want = colname.lower()
        for c in df.columns:
            if c.lower() == want:
                return c
        return None

    px_col = _find_ci("pfx_x")
    pz_col = _find_ci("pfx_z")

    # Keep lowercase copies so downstream 'keep' list always finds them
    if pz_col is not None:
        df["pfx_z"] = pd.to_numeric(df[pz_col], errors="coerce")
        # Vertical: keep sign as-is (positive = ride/up)
        df["INDUCED_VERT_BREAK"] = df["pfx_z"]

    if px_col is not None:
        df["pfx_x"] = pd.to_numeric(df[px_col], errors="coerce")
        # Horizontal: mirror from catcher's → pitcher's view
        # (pitcher's perspective is the x-axis flipped)
        df["HORZ_BREAK"] = -df["pfx_x"]



    # RunValue handling
    if "RunValue" in df.columns:
        df["RunValue"] = pd.to_numeric(df["RunValue"], errors="coerce")
    elif {"RE_before","RE_after","runs_scored_on_pitch"}.issubset(df.columns):
        df["RunValue"] = (
            pd.to_numeric(df["runs_scored_on_pitch"], errors="coerce").fillna(0)
            + pd.to_numeric(df["RE_after"], errors="coerce")
            - pd.to_numeric(df["RE_before"], errors="coerce")
        )
    else:
        df["RunValue"] = np.nan


    # --- ADD: canonicalize plate location cols (from LocationPlots) ---
    def _find_col_ci(df, *names):
        want = {n.lower() for n in names}
        for c in df.columns:
            if c.lower() in want:
                return c
        return None

    rename_map = {}
    pls = _find_col_ci(df, "plate_loc_side", "PlateLocSide", "PLATE_LOC_SIDE", "plate_side", "PLATE_X")
    plh = _find_col_ci(df, "plate_loc_height", "PlateLocHeight", "PLATE_LOC_HEIGHT", "plate_height", "PLATE_Z")
    if pls and pls != "plate_loc_side":
        rename_map[pls] = "plate_loc_side"
    if plh and plh != "plate_loc_height":
        rename_map[plh] = "plate_loc_height"
    if rename_map:
        df = df.rename(columns=rename_map)
    # force numeric for location + key metrics (because you loaded dtype=str)
    for c in [
        "plate_loc_side", "plate_loc_height",
        "Velo", "SPIN_RATE", "RunValue",
        "INDUCED_VERT_BREAK", "HORZ_BREAK",
        "REL_HEIGHT", "REL_SIDE", "EXTENSION",
        "pfx_x", "pfx_z"
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    keep = [
        "Pitcher","GameDate","DateTime","BatterHand","PitchType","PitchResult",
        "PLAY_RESULT","Velo","Count","RunValue",
        "REL_HEIGHT","REL_SIDE","EXTENSION",
        "INDUCED_VERT_BREAK","HORZ_BREAK","SPIN_RATE",
        "EXIT_VELO","LAUNCH_ANGLE",
        "AT_BAT_NUMBER","PITCH_NUMBER_AB","PITCH_NUMBER_GAME",
        "Balls_int","Strikes_int",
        "Balls_post","Strikes_post","Count_post",
        "Balls_pre","Strikes_pre","Count_pre",
        "RE_before","RE_after",
        "pfx_x","pfx_z",
        "plate_loc_side","plate_loc_height",
    ]


    have = [c for c in keep if c in df.columns]
    df["is_unclassified"] = df["PitchType"].isna()

    # canonical order: by game pitch # if available; else DateTime
    if "PITCH_NUMBER_GAME" in df.columns and df["PITCH_NUMBER_GAME"].notna().any():
        df = df.sort_values(["GameDate","PITCH_NUMBER_GAME","AT_BAT_NUMBER","PITCH_NUMBER_AB"], na_position="last")
    else:
        df = df.sort_values("DateTime", na_position="last")

    # convenience: AB id (truth first, fallback if missing)
    if "AT_BAT_NUMBER" in df.columns and df["AT_BAT_NUMBER"].notna().any():
        df["ab_id"] = df["AT_BAT_NUMBER"].astype("Int64")
    else:
        # fallback: rising counter at each 0-0 (legacy)
        df["ab_id"] = (df["Count"] == "0-0").cumsum().astype("Int64")

    df = add_rate_flags(df)

    df = df.reset_index(drop=True)
    df["_row_id"] = df.index.astype(int)
    return df


def _group_by_ab(df: pd.DataFrame):
    """Group in true PA order using your new fields, else legacy 0-0 fallback."""
    if "AT_BAT_NUMBER" in df.columns and df["AT_BAT_NUMBER"].notna().any():
        # robust game → AB → pitch order
        if "PITCH_NUMBER_GAME" in df.columns and df["PITCH_NUMBER_GAME"].notna().any():
            df = df.sort_values(["GameDate","PITCH_NUMBER_GAME","AT_BAT_NUMBER","PITCH_NUMBER_AB"])
        elif "PITCH_NUMBER_AB" in df.columns and df["PITCH_NUMBER_AB"].notna().any():
            df = df.sort_values(["GameDate","AT_BAT_NUMBER","PITCH_NUMBER_AB"])
        else:
            df = df.sort_values(["GameDate","AT_BAT_NUMBER","DateTime"])
        return df.groupby(["GameDate","AT_BAT_NUMBER"], sort=False)
    else:
        # legacy fallback: split PAs by 0-0
        df = df.sort_values("DateTime")
        df["ab_id"] = (df["Count"]=="0-0").cumsum()
        return df.groupby("ab_id", sort=False)


# ---------- 3A. entropy & max-transition  ----------
def predictability_scores(df):
    # next-pitch distribution across ALL ABs, using truth order
    nxt_types = []
    for _, ab in _group_by_ab(df):
        ab = ab.sort_values("PITCH_NUMBER_AB") if "PITCH_NUMBER_AB" in ab.columns and ab["PITCH_NUMBER_AB"].notna().any() else ab.sort_values("DateTime")
        nxt = ab["PitchType"].shift(-1)
        nxt_types.extend(nxt.dropna().tolist())
    total = len(nxt_types)
    if total < 20:
        return {'entropy': None, 'max_prob': None}
    cnt = Counter(nxt_types)
    probs = [v/total for v in cnt.values()]
    H = -sum(p*math.log2(p) for p in probs)
    H_max = math.log2(len(probs))
    return {'entropy': round((1 - H/H_max) * 100, 1), 'max_prob': round(max(probs) * 100, 1)}

# ---------- 3B. colour helper for badge ----------
def colour_for_entropy(e):
    if e is None: return "#7f8c8d"          # grey for “n<20”
    if e < 33:   return "#2ecc71"           # green   (hard to guess)
    if e < 66:   return "#f1c40f"           # amber   (so-so)
    return "#e74c3c"                        # red     (predictable)


# LOCATION PLOT HELPERS

def int_count_to_str(x):
    """Normalize a 'count' value into 'B-S' (e.g., '1-2')."""
    try:
        s = str(x).strip()
        if "-" in s:
            b_str, s_str = s.split("-", 1)
            b, st = int(b_str), int(s_str)
        else:
            n = int(float(s))
            b, st = (divmod(n, 10) if n >= 10 else (0, n))
        if 0 <= b <= 3 and 0 <= st <= 2:
            return f"{b}-{st}"
    except Exception:
        pass
    return None

def _find_col(df, *candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def _compute_in_zone(df):
    """Boolean InZone (independent of catcher view)."""
    if "plate_loc_side" in df.columns and "plate_loc_height" in df.columns:
        in_x = (df["plate_loc_side"] >= X_MIN) & (df["plate_loc_side"] <= X_MAX)
        in_z = (df["plate_loc_height"] >= Z_BOT) & (df["plate_loc_height"] <= Z_TOP)
        df["InZone"] = (in_x & in_z)
    else:
        df["InZone"] = False
    return df

# Robust CSW flag from PitchResult variants
_CSW_CALLED = {
    "called strike", "strike called", "strike (looking)", "strike looking",
    "called_strike", "strike_called"
}
# we'll match substrings for swinging strike types
_SWING_PATTERNS = [
    "swinging strike", "swinging_strike", "swstr", "whiff", "swinging",
    "missed swing", "miss"
]

def _compute_csw(df):
    """Add boolean CSW column using common PitchResult labels."""
    col = _find_col(df, "PitchResult", "PitchCall", "pitch_result", "pitch_call")
    if not col:
        df["CSW"] = False
        return df

    pr = df[col].astype(str).str.strip().str.lower()
    called = pr.isin(_CSW_CALLED)
    swing = False
    for pat in _SWING_PATTERNS:
        swing = swing | pr.str.contains(pat, na=False)
    # exclude fouls (typical CSW doesn’t include fouls)
    is_foul = pr.str.contains("foul", na=False)
    df["CSW"] = (called | swing) & (~is_foul)
    return df

def _compute_miss_oz(df):
    """
    Miss distance from zone center for pitches OUTSIDE zone only (feet).
    Inside-zone rows get NaN so .mean() uses only OZ.
    """
    if {"plate_loc_side", "plate_loc_height", "InZone"}.issubset(df.columns):
        dx = df["plate_loc_side"].astype(float)  # x center = 0
        dy = df["plate_loc_height"].astype(float) - Z_YCTR
        dist = np.sqrt(dx*dx + dy*dy)
        df["MissOZ"] = np.where(df["InZone"], np.nan, dist)
    else:
        df["MissOZ"] = np.nan
    return df

def _ensure_metrics(df):
    """Make sure InZone, CSW, MissOZ exist."""
    if "InZone" not in df.columns:
        df = _compute_in_zone(df)
    if "CSW" not in df.columns:
        df = _compute_csw(df)
    if "MissOZ" not in df.columns:
        df = _compute_miss_oz(df)
    return df

def parse_contents(contents: str, filename: str) -> pd.DataFrame:
    """
    Decode the uploaded file (CSV or Excel), coerce/rename key fields,
    ensure Count is present & normalized, and return cleaned DataFrame.
    Rows with missing Count are dropped.
    """
    content_type, content_string = contents.split(",")
    data = base64.b64decode(content_string)
    buf = io.BytesIO(data)

    # Load CSV vs Excel
    head = data[:128].decode("latin1", errors="ignore")
    is_csv = ("," in head) or filename.lower().endswith(".csv")
    df = pd.read_csv(buf) if is_csv else pd.read_excel(buf)

    # Canonicalize column names
    rename_map = {}
    def map_if_present(df, choices, target):
        src = _find_col(df, *choices)
        if src and src != target:
            rename_map[src] = target

    map_if_present(df, ["plate_loc_side", "PlateLocSide", "PLATE_LOC_SIDE", "plate_side"], "plate_loc_side")
    map_if_present(df, ["plate_loc_height","PlateLocHeight","PLATE_LOC_HEIGHT","plate_height"], "plate_loc_height")
    map_if_present(df, ["Count","COUNT","RawCount","RAW_COUNT","pitch_count"], "Count")
    map_if_present(df, ["PitchType","PITCH_TYPE","pitch_type"], "PitchType")
    map_if_present(df, ["PitchResult","PITCH_RESULT","pitch_result","PitchCall","PITCH_CALL"], "PitchResult")
    map_if_present(df, ["Velo","VELOCITY","Velocity","pitch_velocity"], "Velo")
    map_if_present(df, ["GameDate","GAME_DATE","Date","DATE"], "GameDate")
    map_if_present(df, ["BatterHand","BATTER_HAND","BATTER_SIDE","BatterSide"], "BatterHand")

    if rename_map:
        df = df.rename(columns=rename_map)

    # Normalize Count
    if "Count" not in df.columns:
        raw = _find_col(df, "RawCount", "COUNT", "raw_count")
        if raw:
            df["Count"] = df[raw].apply(int_count_to_str)
        else:
            bcol = _find_col(df, "balls", "Balls")
            scol = _find_col(df, "strikes", "Strikes")
            if bcol and scol:
                df["Count"] = (df[bcol].astype(int).astype(str) + "-" +
                               df[scol].astype(int).astype(str))
            else:
                df["Count"] = None

    df["Count"] = df["Count"].apply(int_count_to_str)
    df = df.dropna(subset=["Count"]).copy()

    # Numerics
    for c in ["plate_loc_side", "plate_loc_height", "Velo"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Date
    if "GameDate" in df.columns:
        df["GameDate"] = pd.to_datetime(df["GameDate"], errors="coerce")

    # required columns present?
    if ("plate_loc_side" not in df.columns) or ("plate_loc_height" not in df.columns):
        raise ValueError("Missing required columns 'plate_loc_side' and/or 'plate_loc_height'.")

    # stable id
    df = df.reset_index(drop=True)
    df["_row_id"] = np.arange(len(df))

    # metrics
    df = _compute_in_zone(df)
    df = _compute_csw(df)
    df = _compute_miss_oz(df)

    return df

def build_scatter(df: pd.DataFrame, color_by: str | None, catcher_view: bool, show_zone: bool):
    plot_df = df.copy()
    if catcher_view and "plate_loc_side" in plot_df.columns:
        plot_df["plate_loc_side"] = -plot_df["plate_loc_side"]

    # Decide color behavior
    color_col = None
    use_datetime_color = False
    if color_by and color_by in plot_df.columns:
        if pd.api.types.is_datetime64_any_dtype(plot_df[color_by]):
            # make a numeric proxy so the scale is continuous oldest→newest
            plot_df["_color_numeric"] = plot_df[color_by].view("int64") // 10**9  # seconds since epoch
            color_col = "_color_numeric"
            use_datetime_color = True
        else:
            color_col = color_by

    fig = px.scatter(
        plot_df,
        x="plate_loc_side",
        y="plate_loc_height",
        color=color_col if color_col else None,
        hover_data=[c for c in plot_df.columns if c not in {"plate_loc_side", "plate_loc_height"}],
        opacity=0.7,
        custom_data=["_row_id"]
    )

    # strike zone overlay
    if show_zone:
        fig.add_shape(
            type="rect",
            x0=X_MIN, x1=X_MAX, y0=Z_BOT, y1=Z_TOP,
            line=dict(color="black", width=2, dash="dot"),
            fillcolor="rgba(0,0,0,0)"
        )

    # If using datetime color, set colorbar ticks to show actual dates + clamp range
    if use_datetime_color:
        cmin = float(plot_df["_color_numeric"].min())
        cmax = float(plot_df["_color_numeric"].max())
        cmid = (cmin + cmax) / 2.0
        tickvals = [cmin, cmid, cmax]
        ticktext = [pd.to_datetime(v, unit="s").strftime("%Y-%m-%d") for v in tickvals]
        fig.update_coloraxes(
            cmin=cmin, cmax=cmax,
            colorbar=dict(title="Oldest → Newest", tickvals=tickvals, ticktext=ticktext)
        )

    fig.update_layout(
        title="Plate Location (Scatter)",
        xaxis_title="plate_loc_side",
        yaxis_title="plate_loc_height",
        legend_title=color_by if (color_by and not use_datetime_color) else "Legend",
        margin=dict(l=40, r=20, t=50, b=40),
        dragmode="lasso"
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    return fig

def build_heatmap(df: pd.DataFrame, catcher_view: bool, show_zone: bool):
    plot_df = df.copy()
    if catcher_view and "plate_loc_side" in plot_df.columns:
        plot_df["plate_loc_side"] = -plot_df["plate_loc_side"]

    fig = px.density_heatmap(
        plot_df,
        x="plate_loc_side",
        y="plate_loc_height",
        nbinsx=60,
        nbinsy=60,
        histfunc="count",
        color_continuous_scale=None
    )

    if show_zone:
        fig.add_shape(
            type="rect",
            x0=X_MIN, x1=X_MAX, y0=Z_BOT, y1=Z_TOP,
            line=dict(color="black", width=2, dash="dot"),
            fillcolor="rgba(0,0,0,0)"
        )

    fig.update_layout(
        title="Plate Location (Heatmap)",
        xaxis_title="plate_loc_side",
        yaxis_title="plate_loc_height",
        margin=dict(l=40, r=20, t=50, b=40),
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    return fig

def make_summary_only(df_view: pd.DataFrame) -> (list[dict], list[dict]):
    """
    Build a TWO-ROW table (TOTALS, AVERAGES) from the final visible set.
    Numeric cols are aggregated; we always add N, ZonePct, CSWPct, MissOZ_AvgFt.
    """
    df_view = df_view.copy()
    df_view = _ensure_metrics(df_view)

    # metrics
    n = len(df_view)
    zone_pct = float(df_view["InZone"].mean() * 100.0) if n else 0.0
    csw_pct  = float(df_view["CSW"].mean() * 100.0) if n else 0.0
    miss_avg = float(df_view["MissOZ"].mean()) if n else float("nan")

    # figure out numeric columns to sum/avg (exclude internal)
    numeric_cols = [c for c in df_view.columns
                    if pd.api.types.is_numeric_dtype(df_view[c])
                    and c not in {"_row_id", "InZone", "CSW", "MissOZ"}]

    totals = {"Summary": "TOTALS", "N": n,
              "ZonePct": round(zone_pct, 2),
              "CSWPct": round(csw_pct, 2),
              "MissOZ_AvgFt": round(miss_avg, 3) if not math.isnan(miss_avg) else None}
    avgs   = {"Summary": "AVERAGES", "N": n,
              "ZonePct": round(zone_pct, 2),
              "CSWPct": round(csw_pct, 2),
              "MissOZ_AvgFt": round(miss_avg, 3) if not math.isnan(miss_avg) else None}

    for c in numeric_cols:
        s = df_view[c].sum(skipna=True)
        m = df_view[c].mean(skipna=True)
        totals[c] = float(s) if pd.notna(s) else None
        avgs[c]   = float(m) if pd.notna(m) else None

    # order: fixed summary fields first, then numeric cols (optional)
    display_cols = ["Summary", "N", "ZonePct", "CSWPct", "MissOZ_AvgFt"] + numeric_cols
    columns = [{"name": c, "id": c} for c in display_cols]
    data = [totals, avgs]
    return columns, data


# ────────────────────────────────────────────────────────────────────
# 4) BUILD PLINKO
# ────────────────────────────────────────────────────────────────────
swing_results = {
    "Foul","Foul Tip","Swinging Strike","Swinging Strike (Blocked)",
    "In Play, No Out","In Play, Out(s)","In Play, Run(s)",
    # add common bunt swing outcomes
    "Foul Bunt","Missed Bunt","Bunt Foul Tip"
}

def swing_pct(df):
    return df.groupby("Count")["PitchResult"] \
             .apply(lambda s: s.isin(swing_results).mean()).to_dict()
def avg_velo(df):
    return df.groupby("Count")["Velo"].mean().to_dict()


def build_plinko(df, mode="frequency", pitch_filter=None, height=None):
    """
    mode ∈ {"frequency","csw","re_delta"}
      - "frequency": keep your tri-palette (pitcher green / hitter red / even gray) by count;
                     intensity by frequency (unchanged).
      - "csw":       color nodes by CSW% at that *pre-pitch* count (numeric colorscale + colorbar).
      - "re_delta":  keep tri-palette hue by count class, but intensity = how much this node
                     typically moves run expectancy on the **next pitch**:
                         E[ΔRE | u] = Σ_v P(v|u) * ( RE(v) − RE(u) )
                     with P derived from observed transitions; RE taken from league (upload),
                     terminals (K/BB) use league RE_after.
    """
    if pitch_filter:
        pitch_filter = pitch_filter.strip().upper()

    # fixed grid
    pos = {
        "0-0":(0,3),"1-0":(1,2),"2-0":(2,1),"3-0":(3,0),
        "0-1":(-1,2),"1-1":(0,1),"2-1":(1,0),"3-1":(2,-1),
        "0-2":(-2,1),"1-2":(-1,0),"2-2":(0,-1),"3-2":(1,-2),
        "K":(-1,-3),"BB":(2,-3),
    }
    end_play = {"In Play, No Out", "In Play, Out(s)", "In Play, Run(s)"}

    df = df.copy()
    df["PitchType"]   = df["PitchType"].astype(str).str.strip()
    df["PitchResult"] = df["PitchResult"].astype(str).str.strip()

    # rows used for node statistics (respect pitch_type filter for the *nodes*)
    rows_for_nodes = df if not pitch_filter else df[df["PitchType"].str.upper() == pitch_filter]
    rows_for_nodes = rows_for_nodes.dropna(subset=["Count"])

    # ---------- per-count numerics for "csw" ----------
    rows_for_nodes = rows_for_nodes.copy()
    rows_for_nodes["csw_flag"] = rows_for_nodes["PitchResult"].isin(CSW_SET).astype(float)
    # ensure Count is a string like "0-0", but the mean is over a numeric column
    rows_for_nodes["Count_for_nodes"] = rows_for_nodes["Count"].fillna(rows_for_nodes["Count_pre"])
    csw_by_cnt = (rows_for_nodes
                  .groupby("Count_for_nodes", dropna=False)["csw_flag"]
                  .mean()
                  .to_dict())

    # Totals per node (where the pitch was thrown) and how many were put in play there
    total_by_pre  = rows_for_nodes.groupby("Count_for_nodes").size().to_dict()
    inplay_by_pre = (
        rows_for_nodes[rows_for_nodes["PitchResult"].isin(end_play)]
        .groupby("Count_for_nodes").size()
    ).to_dict()


    # ---------- transitions P(v|u) ----------
    trans, out_cnt = {}, {}
    node_total = Counter()   # how many pitches thrown at node u
    node_ipno  = Counter()   # how many of those were in-play (end the PA)

    for (_, _ab), ab_full in _group_by_ab(df):
        ab = (ab_full.sort_values("PITCH_NUMBER_AB")
              if "PITCH_NUMBER_AB" in ab_full.columns and ab_full["PITCH_NUMBER_AB"].notna().any()
              else ab_full.sort_values("DateTime")).reset_index(drop=True)

        def post_cnt(r):
            return f"{int(r['Balls_int'])}-{int(r['Strikes_int'])}"

        for i, row in ab.iterrows():
            u = "0-0" if i == 0 else post_cnt(ab.iloc[i-1])  # ← same u as edges
            pr = str(row["PitchResult"])
            b_post = int(row.get("Balls_post") or 0)
            s_post = int(row.get("Strikes_post") or 0)
            play_lc = str(row.get("PLAY_RESULT","")).strip().lower()

            if u not in pos:
                continue

            node_total[u] += 1
            is_ip = pr in end_play
            if is_ip:
                node_ipno[u] += 1

            if is_ip:
                continue
            elif (s_post >= 3 or "strikeout" in play_lc) and u in {"0-2", "1-2", "2-2", "3-2"}:
                v = "K"
            elif (b_post >= 4 or "walk" in play_lc) and u in {"3-0", "3-1", "3-2"}:
                v = "BB"
            else:
                v = post_cnt(row)

            if v not in pos:
                continue

            rec = trans.setdefault((u, v), {"total": 0, "pt": Counter()})
            rec["total"] += 1
            pt = str(row.get("PitchType","")).strip()
            if pt:
                rec["pt"][pt] += 1
            out_cnt[u] = out_cnt.get(u, 0) + 1


    # ---------- E[ΔRE | u] ----------
    # League RE for counts; terminals K/BB from LG_RE_TERM; if missing, we fall back to
    # a neutral zero so nodes don’t blow up.
    RE = LG_RE_BY_COUNT or {}
    RE_term_K  = LG_RE_TERM.get("K")
    RE_term_BB = LG_RE_TERM.get("BB")

    def re_of(node):
        if node in ("K","BB"):
            return RE_term_K if node == "K" else RE_term_BB
        return RE.get(node, None)

    # expected delta by node (only for u with any outgoing transitions)
    exp_delta = {}
    for u in out_cnt.keys():
        RE_u = re_of(u)
        if RE_u is None:
            exp_delta[u] = None
            continue
        total = float(out_cnt[u])
        s = 0.0
        for (uu, v), info in trans.items():
            if uu != u:
                continue
            RE_v = re_of(v)
            if RE_v is None:
                continue
            p = info["total"] / total
            s += p * (RE_v - RE_u)
        exp_delta[u] = s

    # ---------- metric selection per node ----------
    valid_nodes = list(pos.keys())
    node_x, node_y, node_color, node_text, node_hover = [], [], [], [], []

    # helper: tri-palette by count class (already defined in THEME)
    def class_scale(cnt):
        cls = count_class(cnt)
        return THEME["node_pitcher"] if cls == "pitcher" else (
               THEME["node_hitter"]  if cls == "hitter"  else THEME["node_even"])

    # frequency baseline for intensity (counts = pitches thrown at that count)
    freq_by_cnt = {**node_total}
    freq_by_cnt["K"]  = sum(info["total"] for (uu, vv), info in trans.items() if vv == "K")
    freq_by_cnt["BB"] = sum(info["total"] for (uu, vv), info in trans.items() if vv == "BB")
    max_freq = max(1, max((freq_by_cnt.get(k, 0) or 0) for k in pos))

    def clamp01(x): return float(np.clip(x, 0.0, 1.0))
    NEUTRAL = "rgba(235,235,235,1)"  # true neutral for n=0 / n/a
    DELTA_CLIP = 0.05                # for ΔRE display clamp

    for n in sorted(pos, key=lambda k: pos[k][1], reverse=True):
        x, y = pos[n]
        node_x.append(x); node_y.append(y)
        node_text.append(n)

        # Per-node tallies (for header text)
        if n in ("K","BB"):
            total_n = int(freq_by_cnt.get(n, 0))
            header  = f"Terminal: {n} • Arrivals: {total_n}"
        else:
            total_n  = int(node_total.get(n, 0))
            inplay_n = int(node_ipno.get(n, 0))
            trans_n  = int(out_cnt.get(n, 0))   # matches edges by construction
            header   = (f"Pitches at this count: {total_n} • "
                        f"In play: {inplay_n} • Transitions: {trans_n}")

        # ───────── MODE: frequency ─────────
        if mode == "frequency":
            if total_n == 0:
                node_color.append(NEUTRAL)
                node_hover.append(header + "<br>n=0")
            else:
                # small floor so nonzero is visible; ramps start at white
                p = clamp01(total_n / max_freq)
                t = 0.05 + 0.95 * (p ** 0.55)      # gamma < 1 → brighter mid-tones
                rgb = sample_colorscale(class_scale(n), [t])[0]
                node_color.append(rgb.replace("rgb(", "rgba(").replace(")", ",1)"))
                node_hover.append(header)

        # ───────── MODE: csw (% by count) ─────────
        elif mode == "csw":
            if n in ("K","BB"):
                # terminals don’t have a meaningful CSW — keep neutral
                node_color.append(NEUTRAL)
                node_hover.append(header + "<br>CSW%: n/a")
            else:
                v = csw_by_cnt.get(n, np.nan)  # mean of csw_flag at this count
                if pd.isna(v):
                    node_color.append(NEUTRAL)
                    node_hover.append(header + "<br>CSW%: n/a")
                else:
                    # map 0..1 to our white-anchored class ramp
                    p = float(np.clip(v, 0.0, 1.0))
                    t = 0.05 + 0.95 * (p ** 0.60)      # brighten CSW% mids without blowing out max
                    rgb = sample_colorscale(class_scale(n), [t])[0]
                    node_color.append(rgb.replace("rgb(", "rgba(").replace(")", ",1)"))
                    node_hover.append(header + f"<br>CSW%: {v*100:.1f}%")

        # ───────── MODE: re_delta (E[ΔRE | count]) ─────────
        elif mode == "re_delta":
            # ΔRE is based on league RE and observed transitions
            d = exp_delta.get(n, None) if n not in ("K","BB") else None
            if (d is None) or (pd.isna(d)):
                node_color.append(NEUTRAL)
                node_hover.append(header + "<br>E[RV]: n/a")
            else:
                # Convention: negative ΔRE = good for pitcher.
                # THEME['div'] is Rd–White–Gn, so feeding -d gives green for “good”.
                color = div_colour(-float(d), -DELTA_CLIP, DELTA_CLIP, alpha=1.0)
                node_color.append(color)
                node_hover.append(header + f"<br>E[RV]: {d:+.3f}")

        # ───────── fallback (shouldn’t hit) ─────────
        else:
            node_color.append(NEUTRAL)
            node_hover.append(header)


    # build the node trace once (outside the loop)
    node_tr = go.Scatter(
        x=node_x, y=node_y, mode="markers",
        hoverinfo="text", hovertext=node_hover,
        marker=dict(size=50, color=node_color,
                    line=dict(width=0.5, color="rgba(0,0,0,0.25)"),
                    showscale=False)
    )

    # edges (also outside the loop)
    edge_tr, edge_hover = [], []

    def _format_pt_mix(counter, total, k=6):
        if not counter or total <= 0:
            return ""
        items = counter.most_common()
        lines = []
        for i, (pt, cnt) in enumerate(items[:k]):
            pct = 100.0 * cnt / total
            lines.append(f"{pitch_abbr(pt)} {pct:.0f}% ({cnt})")
        if len(items) > k:
            lines.append(f"+{len(items) - k} more")
        return "<br>".join(lines)

    for (u, v), info in trans.items():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        pct   = info["total"] / max(1, out_cnt.get(u, 0))
        alpha = 0.15 + 0.85 * pct
        rgb   = sample_colorscale(THEME["neutral"], [pct])[0]
        r, g, b = [int(c) for c in rgb.strip("rgb()").split(",")]

        edge_tr.append(go.Scatter(
            x=[x0, x1], y=[y0, y1], mode="lines",
            line=dict(width=1 + 14 * pct, color=f"rgba({r},{g},{b},{alpha:.2f})"),
            hoverinfo="none"
        ))

        total = info.get("total", 0)
        mix_txt = _format_pt_mix(info.get("pt", {}), total, k=6)
        hover_txt = (
            f"<b>{u}→{v}</b><br>Total: {total}"
            + (f"<br>{mix_txt}" if mix_txt else "")
        )

        edge_hover.append(go.Scatter(
            x=[(x0 + x1) / 2], y=[(y0 + y1) / 2], mode="markers",
            marker=dict(size=25, color="rgba(0,0,0,0)"),
            hovertext=hover_txt,
            hoverinfo="text"
        ))

    fig = go.Figure(data=edge_tr + edge_hover + [node_tr])

    # label capsules
    anno = []
    for i, (x, y, label) in enumerate(zip(node_x, node_y, node_text)):
        bg = node_color[i] if mode in ("frequency","re_delta","csw") \
            else sample_colorscale(THEME["seq"], [0.5])[0]
        capsule_bg = "rgba(255,255,255,0.65)"
        anno.append(dict(
            x=x, y=y, yshift=18,
            text=label, showarrow=False,
            bgcolor=capsule_bg, bordercolor="rgba(0,0,0,0)",
            xanchor="center", yanchor="bottom",
            font=dict(color="#111", size=12),   # <-- force dark text
            borderpad=2
        ))



    fig.update_layout(
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        plot_bgcolor="white", hovermode="closest",
        height=height if height is not None else 900,
        showlegend=False, annotations=anno
    )
    return fig


# ────────────────────────────────────────────────────────────────────
# 5) DASH APP & LAYOUT
# ────────────────────────────────────────────────────────────────────
ALL_COUNTS = [f"{b}-{s}" for b in range(4) for s in range(3)]


axis_options = [
    {"label": "Run Value",          "value": "RunValue"},
    {"label": "Release Height",      "value": "REL_HEIGHT"},
    {"label": "Release Side",        "value": "REL_SIDE"},
    {"label": "Extension",           "value": "EXTENSION"},
    {"label": "Velocity",            "value": "Velo"},
    {"label": "Exit Velo",           "value": "EXIT_VELO"},
    {"label": "Launch Angle",        "value": "LAUNCH_ANGLE"},
    {"label": "Induced Vert Break",  "value": "INDUCED_VERT_BREAK"},
    {"label": "Horizontal Break",    "value": "HORZ_BREAK"},
    {"label": "Spin Rate",           "value": "SPIN_RATE"},
    {"label": "Count",               "value": "Count"},
    {"label": "Pitch Type",          "value": "PitchType"},
    {"label": "Game Date",           "value": "GameDate"},
]



app = Dash(
    __name__,
    external_stylesheets=["https://codepen.io/chriddyp/pen/bWLwgP.css"],
    suppress_callback_exceptions=True,
)


# ← LEFT-HAND call-out column
LEFT_COL_DIV = html.Div(
    style={
        # was 360px fixed; widen and make it flexible
        "maxWidth":   "640px",
        "minWidth":   "560px",
        "paddingRight": "24px",
        "display":    "flex",
        "flexDirection": "column",
        "gap":        "20px",
    },
    children=[
        # 1) Early → 2) Get-Me-Over → 3) Putaway  (uniform columns)
        html.H4("Early Counts"),
        dash_table.DataTable(
            id="early-strike-table",
            sort_action="native", sort_mode="single",
            style_cell=DT_CELL, style_header=DT_HEAD, style_table=DT_TABLE,
            style_cell_conditional=RATE_WIDTHS,
        ),

        html.H4("Hitter Counts"),
        dash_table.DataTable(
            id="gmo-table",
            sort_action="native", sort_mode="single",
            style_cell=DT_CELL, style_header=DT_HEAD, style_table=DT_TABLE,
            style_cell_conditional=RATE_WIDTHS,
        ),

        html.H4("Late Counts (Two Strikes)"),
        dash_table.DataTable(
            id="put2-table",
            sort_action="native", sort_mode="single",
            style_cell=DT_CELL, style_header=DT_HEAD, style_table=DT_TABLE,
            style_cell_conditional=RATE_WIDTHS,
        ),

    ],
)


# → RIGHT column: Plinko only (animator removed)
RIGHT_COL_DIV = html.Div(
    style={"minWidth": "0", "width": "100%", "paddingLeft": "0"},
    children=[
        html.Div(
            children=[
                html.H3("Plinko Pitch-Sequence",
                        style={"textAlign": "center", "marginBottom": "8px"}),
                dcc.Graph(id="plinko-graph", style={"width": "100%"}),
            ]
        ),
    ],
)

# → HORIZONTAL STRIP: Put-Away Metrics | Early BIP-Outs | CSW% by reps
PUTAWAY_STRIP_DIV = html.Div(
    style={
        "display": "grid",
        "gridTemplateColumns": "repeat(3, minmax(280px, 1fr))",
        "gap": "16px",
        "alignItems": "start",
        "marginTop": "6px",
    },
    children=[
        html.Div(children=[
            html.H4("Two-Strike Put-Away Metrics"),
            dash_table.DataTable(
                id="putaway-table-left",
                columns=[
                    {"name": "Metric",         "id": "Metric"},
                    {"name": "Pitcher",        "id": "P",       "type": "text"},
                    {"name": "League",           "id": "T",       "type": "text"},
                    {"name": "Δ",              "id": "Diff",    "type": "text"},
                    {"name": "",               "id": "Diff_num"},
                ],
                style_as_list_view=True,
                style_cell={"padding":"4px 6px","fontFamily":"Arial","fontSize":"13px","textAlign":"right"},
                style_header={"fontWeight":"bold","textAlign":"center","backgroundColor":"#f4f6f6","border":"1px solid #d5dbdb"},
                style_data={"border":"1px solid #ecf0f1"},
                style_data_conditional=[
                    {"if": {"column_id": "Diff_num"}, "display": "none"},
                    {"if": {"filter_query": "{Metric} = 'Kill %' && {Diff_num} > 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(39,174,96,0.2)", "color": "#27ae60", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = 'Kill %' && {Diff_num} < 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(192,57,43,0.2)", "color": "#c0392b", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = 'PATS' && {Diff_num} > 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(192,57,43,0.2)", "color": "#c0392b", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = 'PATS' && {Diff_num} < 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(39,174,96,0.2)", "color": "#27ae60", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = 'Avg Fouls 2-Str' && {Diff_num} > 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(192,57,43,0.2)", "color": "#c0392b", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = 'Avg Fouls 2-Str' && {Diff_num} < 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(39,174,96,0.2)", "color": "#27ae60", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = 'Avg Balls 2-Str' && {Diff_num} > 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(192,57,43,0.2)", "color": "#c0392b", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = 'Avg Balls 2-Str' && {Diff_num} < 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(39,174,96,0.2)", "color": "#27ae60", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = '% IP-No-Out 2-Str' && {Diff_num} > 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(192,57,43,0.2)", "color": "#c0392b", "fontWeight": "bold"},
                    {"if": {"filter_query": "{Metric} = '% IP-No-Out 2-Str' && {Diff_num} < 0", "column_id": "Diff"},
                     "backgroundColor": "rgba(39,174,96,0.2)", "color": "#27ae60", "fontWeight": "bold"},
                ],
                tooltip_delay=0, tooltip_duration=None,
                page_action="none",
                style_table={"height":"180px","overflowY":"auto","border":"none"},
            ),
        ]),

        html.Div(children=[
            html.H4("Best Early BIP-Outs"),
            dash_table.DataTable(
                id="early-out-table",
                sort_action="native", sort_mode="single",
                style_cell=DT_CELL, style_header=DT_HEAD, style_table=DT_TABLE,
            ),
        ]),

        html.Div(children=[
            html.H4("Pitch CSW % by # Reps in PA"),
            dash_table.DataTable(
                id="occurrence-table",
                sort_action="native", sort_mode="single",
                tooltip_header={
                    "PitchType":"Pitch type",
                    "1x":"CSW% the 1st time this pitch appears in the PA",
                    "2x":"…the 2nd time",
                    "3x+":"…the 3rd time or later",
                },
                tooltip_delay=0, tooltip_duration=None,
                style_cell=DT_CELL, style_header=DT_HEAD, style_table=DT_TABLE,
            ),
            html.Div(id="pred-score-wrapper",
                     style={"textAlign":"center","marginTop":"6px"}),
        ]),
    ],
)


app.layout = html.Div(style={"fontFamily": "Arial",
                             "maxWidth": "1400px",
                             "margin":    "20px auto"}, children=[

    html.H1("Usage / Sequencing Dashboard", style={"textAlign": "center"}),

    # Upload control + status
    html.Div(
        [
            dcc.Upload(
                id="upload-data",
                children=html.Div(["Drag & Drop or ", html.A("Select a CSV/XLSX")]),
                multiple=False,
                style={
                    "border": "1px dashed #888",
                    "padding": "10px",
                    "textAlign": "center",
                    "margin": "8px auto",
                    "maxWidth": "480px",
                },
            ),
            html.Div(id="upload-status", style={"textAlign": "center", "margin": "6px 0"}),
        ]
    ),

    dcc.Store(id="raw-data"),       # now empty until upload
    dcc.Store(id="filtered-data"),  # holds current subset


    # ── FILTER ROW 1 ────────────────────────────────────────────────
    html.Div(style={"display": "flex", "gap": "10px",
                    "marginBottom": "10px"}, children=[
        dcc.Dropdown(id="pitcher-dd", options=[],
                    placeholder="Select pitcher", style={"flex": 2}),
        dcc.Dropdown(id="date-dd", placeholder="Select dates",
                     multi=True, style={"flex": 3}),
        dcc.Dropdown(id="bat-hand-dd",
                     options=[{"label": "Both",  "value": "All"},
                              {"label": "Left",  "value": "L"},
                              {"label": "Right", "value": "R"}],
                     value="All", style={"flex": 1}),
    ]),

    # ── FILTER ROW 2 ────────────────────────────────────────────────
    html.Div(style={"display": "flex", "gap": "10px",
                    "marginBottom": "10px"}, children=[
        dcc.Dropdown(id="ptype-dd", placeholder="Select pitch type",
                     style={"flex": 3}),
        dcc.RadioItems(
            id="mode",
            options=[
                {"label": "Count Freq",                 "value": "frequency"},
                {"label": "CSW% by Count",              "value": "csw"},
                {"label": "Next-Pitch ΔRE (by Count)",  "value": "re_delta"},
            ],
            value="frequency",
            labelStyle={"display": "inline-block", "marginRight": "10px"},
            style={"flex": 2, "paddingLeft": "10px"},
        ),

        dcc.Dropdown(id="bucket-custom-count",
                     options=[{"label": c, "value": c}
                              for c in ALL_COUNTS],
                     placeholder="Counts for CUSTOM bucket",
                     multi=True, style={"flex": 2}),
    ]),

    # ── MINI COUNT KPI STRIP ────────────────────────────────────────
    html.Div(id="count-summary",
            style={"display": "flex", "justifyContent": "center",
                    "gap": "25px", "flexWrap": "wrap", "margin": "10px 0"}),

    html.Div(
        id="nan-warning",
        style={
            "display": "flex",
            "justifyContent": "center",
            "alignItems": "center",
            "gap": "8px",
            "margin": "4px 0 6px 0",
            "fontSize": "13px",
            "color": "#8a6d3b",
        },
    ),

    html.Div(
        id="pitch-rv-tiles",
        style={
            "display": "flex",
            "gap": "10px",
            "justifyContent": "center",
            "flexWrap": "wrap",
            "marginTop": "0px",
            "marginBottom": "10px",
        },
    ),


  # ── CALL-OUTs + PLINKO ─────────────────────────────────────────────
  html.Div(
      style={
          "display": "grid",
          # was "360px minmax(520px, 1fr)"
          "gridTemplateColumns": "minmax(560px, 42%) 1fr",
          "gap": "24px",
          "alignItems": "start",
          "marginTop": "15px",
          "maxWidth": "1400px",
          "width": "100%",
      },
      children=[LEFT_COL_DIV, RIGHT_COL_DIV],
  ),

  # Side-by-side: Putaway Metrics → Best Early BIP-Outs → CSW% by reps
  PUTAWAY_STRIP_DIV,

  # ── PITCH-TO-PITCH & BUCKET HEATMAP MATRICES ─────────────────────────────
  html.Div(
      style={
          "display": "grid",
          "gridTemplateColumns": "repeat(2, 1fr)",
          "gap": "25px",
          # optional: small top margin so it breathes under the strip
          "marginTop": "20px",
      },
      children=[
          dcc.Graph(id="heat-freq"),
          dcc.Graph(id="heat-csw"),
          dcc.Graph(id="usage-count"),
          dcc.Graph(id="csw-bucket"),
          dcc.Graph(id="velo-count"),
      ],
  ),


# ── METRIC SECTION  (Pitch RV / Mix / Doubledown vs Mix / Fixer) ──────────
html.Div(
    style={
        "display": "grid",
        "gridTemplateColumns": "minmax(260px, 1fr) minmax(360px, 1.5fr) minmax(260px, 1fr)",
        "gap": "14px",
        "alignItems": "stretch",
    },
    children=[

        # 2️⃣ Mix summary (Even-Mix / Repeat / Mix)
        html.Div(
            children=[
                html.H4("Mix / Repeat / Even-Mix"),
                dash_table.DataTable(
                    id="mix-table",
                    columns=[
                        {"name":"Bucket","id":"Bucket"},
                        {"name":"Mix %","id":"MixPct"},
                        {"name":"Repeat %","id":"RepeatPct"},
                        {"name":"Even-Mix %","id":"BlendPct"},
                    ],
                    style_as_list_view=True,
                    style_header={"fontWeight":"bold"},
                    tooltip_delay=0, tooltip_duration=None,
                    style_data_conditional=[
                        {"if":{"column_id":"BlendPct"},
                         "backgroundColor":"rgba(39,174,96,0.2)",
                         "padding":"0 4px"},
                        {"if":{"column_id":"RepeatPct"},
                         "backgroundColor":"rgba(192,57,43,0.2)",
                         "padding":"0 4px"},
                    ],
                    page_action="none",
                    style_table={"height":"300px", "overflowY":"auto"},
                ),
            ],
            style={"minWidth":"0"},
        ),

        # 3️⃣ Doubledown vs Mix dumbbell (wider column)
        html.Div(
            children=[
                html.H4("Doubledown vs Mix"),
                dcc.Graph(id="repeat-mix", style={"height": "340px"}),  # a bit taller
            ],
            style={"minWidth":"0"},
        ),

        # 4️⃣ Fixer table (Get-Me-Right)
        html.Div(
            children=[
                html.H4("Fixer (Get-Me-Right)"),
            dash_table.DataTable(
                id="gmr-table",
                columns=[
                    {"name": "Prev Pitch",  "id": "PrevPitch"},
                    {"name": "Helper",      "id": "HelperPitch"},
                    {"name": "Success",     "id": "Success"},
                    {"name": "Attempts",    "id": "Attempts"},
                    {"name": "Success %",   "id": "SuccessRate"},
                ],
                style_as_list_view=True,
                tooltip_delay=0,
                tooltip_duration=None,
                style_header={"fontWeight": "bold"},
                style_data_conditional=[
                    {"if": {"column_id": "SuccessRate"},
                     "backgroundColor": "rgba(52,152,219,0.18)",
                     "padding": "0 4px"},
                ],
                sort_action="native",
                sort_mode="single",
                page_action="none",
                style_table={"height": "300px", "overflowY": "auto"},
            ),
            ],
            style={"minWidth":"0"},
        ),
    ],
),


    # ── COUNT FILTER SWITCH ─────────────────────────────────────────
    html.Div([
        html.Label("Count Filter:", style={"fontWeight":"bold", "marginRight":"10px"}),
        dcc.RadioItems(
            id="count-filter",
            options=[
                {"label": "Overall", "value": "Overall"},
                {"label": "Early",   "value": "Early"},
                {"label": "Late",    "value": "Late"},
            ],
            value="Overall",
            inline=True,
        ),
    ], style={"marginBottom": "20px"}),

    # ── USAGE LINE (bottom) ────────────────────────────────────────
    dcc.Graph(id="usage-line"),

    # ── MIXED-STACK + 7-DAY CSW ────────────────────────────────────
    html.H2("Pitch-Mix Scaled to Rolling CSW%", style={"marginTop": "40px"}),
    dcc.Graph(id="mix-csw"),

    # ── 7-DAY ROLLING AVERAGES CHART ───────────────────────────────
    html.H2("7-Day Rolling Averages Chart", style={"marginTop": "40px"}),

    html.Div(style={"display": "flex", "gap": "10px",
                    "flexWrap": "wrap"}, children=[
        dcc.Dropdown(id="rolling-metric",
                     value="RollingUsage",
                     options=[{"label": "7-Day Rolling Usage %", "value": "RollingUsage"},
                              {"label": "7-Day Rolling Velo",     "value": "RollingVelo"},
                              {"label": "7-Day Rolling CSW %",   "value": "RollingCSW"}],
                     style={"width": "220px"}),
        dcc.Dropdown(id="cust-count",
                     placeholder="Filter counts (multi)",
                     multi=True, style={"minWidth": "160px"}),
        dcc.Dropdown(id="cust-ptype",
                     placeholder="Filter pitch types (multi)",
                     multi=True, style={"minWidth": "180px"}),
    ]),
        dcc.Checklist(
            id="rolling-overall-csw",
            options=[{"label":"Show overall 7-day CSW% line","value":"csw7"}],
            value=[],  # off by default
            style={"alignSelf":"center"}
        ),
    dcc.Graph(id="rolling-graph"),

    # ── NEW: CSW% vs X Scatter Plot ─────────────────────────────────
    html.Div(                              # <— replace the old one with this
        style={"display": "flex", "gap": "10px", "flexWrap": "wrap"},
        children=[
            dcc.Dropdown(
                id="scatter-x", options=axis_options,
                value="HORZ_BREAK",                    # ← default X
                placeholder="X-axis",
                style={"width": "180px"}
            ),

            dcc.Dropdown(
                id="scatter-y", options=axis_options,
                value="INDUCED_VERT_BREAK",            # ← default Y
                placeholder="Y-axis",
                style={"width": "180px"}
            ),

            dcc.Dropdown(id="scatter-color", options=axis_options,
                        value="SPIN_RATE", placeholder="Colour",
                        style={"width": "180px"}),

            dcc.Dropdown(id="scatter-ptype", multi=True,
                        placeholder="Filter pitch types…",
                        style={"minWidth": "200px"}),
            dcc.RadioItems(
                id="scatter-agg",
                options=[{"label": "Averages", "value": "avg"},
                        {"label": "Every pitch", "value": "raw"}],
                value="avg",
                labelStyle={"display": "inline-block", "marginRight": "12px"},
            ),
        ]),

    html.Div(
        style={
            "display": "grid",
            "gridTemplateColumns": "repeat(2, 1fr)",
            "gap": "16px",
            "alignItems": "stretch",
            "marginTop": "8px",
        },
        children=[
            # left half — the existing scatter (now sized square-ish)
            dcc.Graph(id="csw-scatter", style={"width": "100%", "height": "620px"}),

            # right half — location plot
            html.Div(
                style={"display":"flex","flexDirection":"column","gap":"8px"},
                children=[
                    html.Div(
                        style={"display":"flex","gap":"10px","flexWrap":"wrap","alignItems":"center"},
                        children=[
                            dcc.Dropdown(id="loc-color-by", placeholder="Colour by…", style={"width":"220px"}),
                            dcc.RadioItems(
                                id="loc-plot-type",
                                options=[{"label":" Scatter","value":"scatter"},{"label":" Heatmap","value":"heatmap"}],
                                value="scatter",
                                inline=True,
                                style={"marginLeft":"10px"},
                            ),
                            dcc.Checklist(
                                id="loc-flags",
                                options=[
                                    {"label":" Catcher view (flip X)", "value":"flip"},
                                    {"label":" Show generic strike zone", "value":"zone"},
                                ],
                                value=["zone"],
                                inline=True,
                                style={"marginLeft":"10px"},
                            ),
                            html.Button("Clear selection", id="loc-clear-selection", n_clicks=0),
                            html.Div(id="loc-zone-metric", style={"marginLeft":"auto","fontWeight":"bold"}),
                        ],
                    ),
                    dcc.Graph(id="loc-plate-graph", style={"height":"620px"}),
                    dcc.Store(id="loc-selected-ids"),
                ],
            ),
        ],
    ),


    html.H3("Location Summary (current filters)"),
    dash_table.DataTable(
        id="loc-summary-table",
        style_as_list_view=True,
        fixed_rows={"headers": True, "data": 0},
        style_table={"overflowX":"auto"},
        style_cell={"padding":"4px","minWidth":90,"maxWidth":320,"whiteSpace":"normal"},
        style_header={"fontWeight":"bold"},
    ),
    html.H3("Pitch table (filters + selection drive the plate plot)"),
    dash_table.DataTable(
        id="loc-table",
        filter_action="native",
        sort_action="native",
        page_action="native",
        page_size=25,
        row_selectable=False,
        style_table={"height":"420px","overflowY":"auto","overflowX":"auto"},
        style_cell={"padding":"4px","minWidth":80,"maxWidth":260,"whiteSpace":"normal"},
        style_header={"fontWeight":"bold"},
    ),


])


# ────────────────────────────────────────────────────────────────────
# 6) CALLBACKS
# ────────────────────────────────────────────────────────────────────


@app.callback(Output("date-dd","options"),
              Input("pitcher-dd","value"),
              State("raw-data","data"))
def set_dates(pitcher,data):
    if not (pitcher and data): return []
    df = pd.read_json(data, orient="split")
    df["GameDate"] = pd.to_datetime(df["GameDate"]).dt.date
    ds = sorted(df[df["Pitcher"]==pitcher]["GameDate"].unique())
    return [{"label":d.isoformat(),"value":d.isoformat()} for d in ds]

@app.callback(Output("ptype-dd","options"),
              Input("raw-data","data"),
              Input("pitcher-dd","value"),
              Input("date-dd","value"))
def set_ptypes(data,pitcher,dates):
    if not data: return []
    df = pd.read_json(data, orient="split")
    df["GameDate"] = pd.to_datetime(df["GameDate"]).dt.date
    if pitcher: df = df[df["Pitcher"]==pitcher]
    if dates:
        sel = [pd.to_datetime(d).date() for d in dates]
        df = df[df["GameDate"].isin(sel)]
    pts = sorted(df["PitchType"].dropna().unique())
    return [{"label": pt, "value": pt} for pt in pts]


# ---------- FIX update_plinko ---------- #
@app.callback(
    Output("filtered-data", "data"),
    Input("raw-data",   "data"),
    Input("pitcher-dd", "value"),
    Input("date-dd",    "value"),
    Input("bat-hand-dd","value"),
    Input("ptype-dd",   "value"),
)
def make_subset(raw_json, pitcher, dates, hand, ptype):
    if not (raw_json and pitcher):
        return None

    df = pd.read_json(raw_json, orient="split")

    # 👉  make the types match
    df["GameDate"] = pd.to_datetime(df["GameDate"]).dt.date

    df = df[df["Pitcher"] == pitcher]

    if dates:                                    # → filter by date(s)
        sel = [pd.to_datetime(d).date() for d in dates]
        df  = df[df["GameDate"].isin(sel)]

    if hand != "All":
        df = df[df["BatterHand"] == hand]

    if df.empty:
        return None
    return df.to_json(date_format="iso", orient="split")

# ---------- FIX update_plinko height target ---------- #
@app.callback(
    Output("plinko-graph", "figure"),
    Output("pred-score-wrapper", "children"),
    Input("filtered-data", "data"),
    Input("ptype-dd",      "value"),
    Input("mode",          "value"),
    Input("early-strike-table", "data"),
    Input("gmo-table",          "data"),
    Input("put2-table",         "data"),
)
def update_plinko(sub_json, ptype, mode, early_rows, gmo_rows, put_rows):
    if not sub_json:
        return go.Figure(), "Select a pitcher to start"

    df = pd.read_json(sub_json, orient="split")

    # --- dynamic height aimed at end of Putaway table --------------
    # tweak base/per_row if your font or row padding differs
    n_early = len(early_rows) if early_rows else 8
    n_gmo   = len(gmo_rows)   if gmo_rows   else 8
    n_put   = len(put_rows)   if put_rows   else 8

    base_h  = 420   # header + spacers before the first table
    per_row = 28    # typical DataTable row height incl. padding
    dyn_h   = max(800, base_h + per_row * (n_early + n_gmo + n_put))

    fig = build_plinko(df, mode, pitch_filter=ptype, height=dyn_h)

    preds = predictability_scores(df)
    if preds["entropy"] is None:
        badge = html.Span("n<20", style={"background":"#7f8c8d","color":"white","padding":"2px 6px","borderRadius":"4px"})
        txt_children = ["Predictability: ", badge, "   |   Max repeat: (too few pitches)"]
    else:
        badge = html.Span(f"{preds['entropy']:.1f}/100", style={"background": colour_for_entropy(preds['entropy']),"color":"white","padding":"2px 6px","borderRadius":"4px"})
        txt_children = ["Predictability: ", badge, f"   |   Max repeat: {preds['max_prob']:.1f}%"]

    return fig, html.Div(txt_children)


@app.callback(
    Output("raw-data", "data"),
    Output("pitcher-dd", "options"),
    Output("upload-status", "children"),
    Input("upload-data", "contents"),
    State("upload-data", "filename"),
)

def handle_upload(contents, filename):
    if not contents:
        return dash.no_update, dash.no_update, ""

    try:
        _, b64 = contents.split(",", 1)
        data_bytes = base64.b64decode(b64)
    except Exception:
        return dash.no_update, dash.no_update, "Upload failed: could not read file."

    try:
        df = load_pitchlog(data_bytes)
    except Exception as e:
        return dash.no_update, dash.no_update, f"Parsing error: {e}"

    # Coerce likely-numeric columns that sometimes arrive as strings
    num_cols = [
        "RunValue", "RE_before", "RE_after",
        "Balls_post", "Strikes_post"
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # If Count is stored like "0-0" already, keep it. If it's split columns, assemble:
    if "Count" in df.columns:
        # normalize spacing/casing
        df["Count"] = df["Count"].astype(str).str.strip()
    elif {"Balls", "Strikes"} <= set(df.columns):
        df["Count"] = df["Balls"].astype(int).astype(str) + "-" + df["Strikes"].astype(int).astype(str)

    # league references from uploaded data
    global LG_BY_COUNT, LG_PCTL, LG_RE_BY_COUNT, LG_RE_TERM

    lg = df.dropna(subset=["RunValue","Count"]).copy()
    LG_BY_COUNT = lg.groupby("Count")["RunValue"].mean().to_dict()

    by_pt = lg.groupby("PitchType").agg(RV_sum=("RunValue","sum"), N=("RunValue","size"))
    by_pt["RV100"] = 100 * by_pt["RV_sum"] / by_pt["N"]
    LG_PCTL = by_pt["RV100"].rank(pct=True, ascending=False).to_dict()

    # NEW: league run expectancy by count and terminals
    if "RE_before" in df.columns:
        tmp = df.dropna(subset=["RE_before","Count"]).copy()
        LG_RE_BY_COUNT = tmp.groupby("Count")["RE_before"].mean().to_dict()
    else:
        LG_RE_BY_COUNT = {}

    LG_RE_TERM = {"K": None, "BB": None}
    if "RE_after" in df.columns:
        term = df.copy()
        term["play_lc"] = term.get("PLAY_RESULT","").astype(str).str.lower()
        is_k  = (term["Strikes_post"]>=3) | term["play_lc"].str.contains("strikeout", na=False)
        is_bb = (term["Balls_post"]>=4)   | term["play_lc"].str.contains("walk",      na=False)
        if is_k.any():
            LG_RE_TERM["K"]  = term.loc[is_k, "RE_after"].dropna().mean()
        if is_bb.any():
            LG_RE_TERM["BB"] = term.loc[is_bb, "RE_after"].dropna().mean()

    opts = [{"label": p, "value": p} for p in sorted(df["Pitcher"].dropna().unique())]
    status = f"Loaded {len(df):,} pitches from {filename}"
    return df.to_json(date_format="iso", orient="split"), opts, status



def _percentile_frame(df, axis=0):
    # column-wise percentiles (axis=0): each column independently
    # row-wise percentiles (axis=1): each row independently
    arr = df.copy().astype(float)
    if axis == 0:
        for c in arr.columns:
            col = arr[c]
            rank = col.rank(pct=True)
            arr[c] = rank
    else:
        for r in arr.index:
            row = arr.loc[r]
            rank = row.rank(pct=True)
            arr.loc[r] = rank
    return arr


def _leverage(csw_pct_df, usage_pct_df, *, by="column"):
    axis = 0 if by == "column" else 1
    csw_p  = _percentile_frame(csw_pct_df, axis=axis)          # high good
    use_p  = _percentile_frame(usage_pct_df,  axis=axis)        # low good
    lev    = csw_p * (1.0 - use_p)
    return lev.clip(0, 1)

@app.callback(
    Output("heat-freq",   "figure"),
    Output("heat-csw",    "figure"),
    Output("usage-count", "figure"),
    Output("csw-bucket",  "figure"),
    Output("velo-count",  "figure"),
    Input("filtered-data",        "data"),
    Input("ptype-dd",             "value"),
    Input("mode",                 "value"),
    Input("bucket-custom-count",  "value"),
)

def update_heatmaps(sub_json, ptype, mode, custom_sel):
    if not sub_json:
        return [go.Figure()] * 5

    df = pd.read_json(sub_json, orient="split")

    BUCKET_HEATMAP_H = 400

    # ─── drop pitch types <3% overall usage (computed on this subset BEFORE any dropdown filter) ───
    df["_PT_STD"] = _pt_std(df["PitchType"])
    pt = df["_PT_STD"].dropna()
    if pt.empty:
        return [go.Figure().update_layout(title="No tagged pitch types for current filters")] * 5

    usage = pt.value_counts(normalize=True)
    keep = set(usage[usage >= MIN_PITCHTYPE_USAGE_TILE].index)
    df = df[df["_PT_STD"].isin(keep)].copy()

    # ─── optional pitch-type filter ────────────────────────────────
    if ptype:
        p_clean = str(ptype).strip().upper()
        df = df[df["_PT_STD"] == p_clean]
    if df.empty:
        return [go.Figure().update_layout(title="No data for current filters")] * 5

    # No longer needed downstream
    df = df.drop(columns=["_PT_STD"])

    # after filtering, build a GLOBAL csw flag used by multiple blocks
    df["csw"] = df["PitchResult"].isin(CSW_SET).astype(int)

    # ----------------------------------------------------------------
    # 1)  pitch-to-pitch transition (row-normalized freq %  & CSW %)
    # ----------------------------------------------------------------
    rows = []
    for _, ab in _group_by_ab(df):
        ab = ab.sort_values("PITCH_NUMBER_AB") if "PITCH_NUMBER_AB" in ab.columns and ab["PITCH_NUMBER_AB"].notna().any() else ab.sort_values("DateTime")
        ab["csw"] = ab["PitchResult"].isin(CSW_SET).astype(int)
        ab["prev"] = ab["PitchType"].shift()
        rows.append(ab.dropna(subset=["prev"])[["prev", "PitchType", "csw"]])
    if not rows:
        blank = go.Figure().update_layout(
            title="Pitch-to-Pitch Transition % (no transitions)",
            xaxis_visible=False, yaxis_visible=False)
        return [blank, blank, blank, blank, blank]
    trans = pd.concat(rows, ignore_index=True)

    # ───────────────────────────────────────────────
    # 1) Pitch-to-pitch transition % and CSW%
    # ───────────────────────────────────────────────
    freq_cnt = (
        trans.groupby(["prev", "PitchType"])
             .size()
             .unstack(fill_value=0)
    )
    freq_cnt = freq_cnt.loc[freq_cnt.sum(axis=1) > 0, :]
    freq_cnt = freq_cnt.loc[:, freq_cnt.sum(axis=0) > 0]
    if freq_cnt.empty:
        blank = go.Figure().update_layout(
            title="Pitch-to-Pitch Transition % (no data after filtering)",
            xaxis_visible=False,
            yaxis_visible=False,
        )
        return [blank, blank, blank, blank, blank]

    # Row-normalized transition %
    row_den = freq_cnt.sum(axis=1).replace(0, np.nan)
    freq_pct = (freq_cnt.div(row_den, axis=0) * 100).fillna(0).round(1)

    # CSW% for each prev → next combo
    csw_succ = (
        trans.groupby(["prev", "PitchType"])["csw"]
             .sum()
             .unstack(fill_value=0)
             .reindex(index=freq_cnt.index,
                      columns=freq_cnt.columns,
                      fill_value=0)
    )
    with np.errstate(invalid="ignore", divide="ignore"):
        csw_pct = (
            csw_succ / freq_cnt.replace(0, np.nan) * 100
        ).fillna(0).round(1)

    # Shared annotator used by ALL the heatmaps below
    def annotate(df_pct, df_cnt):
        txt = df_pct.copy().astype(str)
        for r in df_pct.index:
            for c in df_pct.columns:
                pct = df_pct.loc[r, c]
                cnt = int(df_cnt.loc[r, c])
                txt.loc[r, c] = f"{pct:.1f}%<br>({cnt})" if cnt else ""
        return txt.values

    # Transition% heatmap (same as before)
    freq_fig = px.imshow(
        freq_pct,
        x=freq_pct.columns.tolist(),
        y=freq_pct.index.tolist(),
        color_continuous_scale=THEME["trans"],
        aspect="auto",
    )
    freq_fig.update_traces(
        text=annotate(freq_pct, freq_cnt),
        texttemplate="%{text}",
        textfont_size=12,
    )
    freq_fig.update_layout(
        title="Pitch-to-Pitch Transition % (row-normalized)",
        margin=dict(l=0, r=0, t=40, b=0),
    )
    freq_fig.update_yaxes(autorange="reversed", ticklabelposition="outside left")

    # ───────────────────────────────────────────────
    # NEW: CSW% vs transition% using medians + SDs
    # green = high CSW & under-transitioned
    # red   = low CSW  & over-transitioned
    # ───────────────────────────────────────────────

    MIN_OVERALL_TRANS = 5.0  # % of ALL transitions for a given next pitch

    # Overall transition share per next pitch type (columns)
    overall_trans_cnt = freq_cnt.sum(axis=0)
    total_transitions = overall_trans_cnt.sum()
    overall_trans_pct = overall_trans_cnt / total_transitions * 100.0

    low_overall = overall_trans_pct < MIN_OVERALL_TRANS  # Series indexed by PitchType

    # Copies for stats: low-overall transitions -> NaN so they don't affect medians/SDs
    csw_stats = csw_pct.copy().astype(float)
    trans_stats = freq_pct.copy().astype(float)
    csw_stats.loc[:, low_overall] = np.nan
    trans_stats.loc[:, low_overall] = np.nan

    # Row-wise medians and SDs for CSW%
    row_median_csw = csw_stats.median(axis=1, skipna=True)
    row_std_csw = csw_stats.std(axis=1, ddof=0, skipna=True).replace(0, np.nan)
    z_csw = csw_stats.sub(row_median_csw, axis=0).div(row_std_csw, axis=0)

    # Row-wise medians and SDs for transition%
    row_median_trans = trans_stats.median(axis=1, skipna=True)
    row_std_trans = trans_stats.std(axis=1, ddof=0, skipna=True).replace(0, np.nan)
    z_trans = trans_stats.sub(row_median_trans, axis=0).div(row_std_trans, axis=0)

    # Decompose into "good / bad" CSW and "under / over" transition
    good_csw = z_csw.clip(lower=0)        # CSW above row median
    bad_csw = (-z_csw).clip(lower=0)      # CSW below row median
    under_tr = (-z_trans).clip(lower=0)   # transition% below row median
    over_tr = z_trans.clip(lower=0)       # transition% above row median

    # +: high CSW & low transition%  → throw this more after that previous pitch
    # -: low CSW  & high transition% → throw this less after that previous pitch
    metric = good_csw * under_tr - bad_csw * over_tr

    # Neutralize NaNs and <5% overall transitions
    metric = metric.fillna(0.0)
    metric.loc[:, low_overall] = 0.0

    # Symmetric color range around 0 (0 = white)
    with np.errstate(all="ignore"):
        max_abs = float(np.nanmax(np.abs(metric.values))) if metric.values.size else 0.0
    if (not np.isfinite(max_abs)) or max_abs == 0.0:
        zmin, zmax = -1.0, 1.0
    else:
        zmin, zmax = -max_abs, max_abs

    csw_fig = px.imshow(
        metric,
        x=csw_pct.columns.tolist(),
        y=csw_pct.index.tolist(),
        color_continuous_scale=THEME["div"],  # same red–white–green as bucket plot
        color_continuous_midpoint=0,
        zmin=zmin,
        zmax=zmax,
        aspect="auto",
    )
    csw_fig.update_traces(
        text=annotate(csw_pct, csw_succ),
        texttemplate="%{text}",
        textfont_size=12,
    )
    csw_fig.update_layout(
        title=(
            "Pitch-to-Pitch CSW% "
            "(green = high CSW & under-transitioned; red = low CSW & over-transitioned)"
        ),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    csw_fig.update_yaxes(autorange="reversed", ticklabelposition="outside left")

    # ----------------------------------------------------------------
    # 2)  BUCKET logic (unchanged; uses CUSTOM selector)
    # ----------------------------------------------------------------
    ROW_ORDER = ["Early Count", "Late Count", "Hitter's",
                 "Pitcher's", "CUSTOM"]
    custom_set = set(custom_sel or [])

    def buckets(ct):
        tag = []
        if ct in EARLY_CT: tag.append("Early Count")
        if ct in LATE_CT: tag.append("Late Count")
        if ct in HITTER_CT: tag.append("Hitter's")
        if ct in PITCHER_CT: tag.append("Pitcher's")
        if ct in custom_set: tag.append("CUSTOM")
        return tag

    # 🔧 Use PRE-pitch count everywhere count-based:
    bucket_count = df.get("Count_pre", df["Count"]).fillna(df["Count"])
    bucket_count = bucket_count.astype(str).str.strip()

    df["BucketList"] = bucket_count.apply(buckets)
    df_exp = df.explode("BucketList").dropna(subset=["BucketList"])
    df_exp["Velo"] = pd.to_numeric(df_exp["Velo"], errors="coerce")

    # Usage % by bucket
    usage_cnt = (df_exp.groupby(["BucketList", "PitchType"])
                       .size()
                       .unstack(fill_value=0)
                       .reindex(ROW_ORDER, fill_value=0))
    usage_pct = usage_cnt.div(usage_cnt.sum(axis=1).replace(0, np.nan), axis=0).mul(100).fillna(0)
    usage_fig = px.imshow(
        usage_pct.round(1),
        x=usage_pct.columns.tolist(), y=usage_pct.index.tolist(),
        color_continuous_scale=THEME["usage"],
        aspect="auto", height=BUCKET_HEATMAP_H
    )

    usage_fig.update_traces(
        text=annotate(usage_pct, usage_cnt),
        texttemplate="%{text}", textfont_size=12)
    usage_fig.update_layout(title="Pitch Usage % by Bucket",
                            margin=dict(l=0, r=0, t=40, b=0))
    usage_fig.update_yaxes(autorange="reversed", ticklabelposition="outside left")

    # CSW % by bucket
    csw_succ_b = (df_exp.groupby(["BucketList", "PitchType"])["csw"]
                         .sum()
                         .unstack(fill_value=0)
                         .reindex(ROW_ORDER, fill_value=0))
    csw_pct_b = (csw_succ_b / usage_cnt.replace(0, np.nan) * 100).fillna(0)

    # ─────────────────────────────────────────────────────────────
    #  NEW: median + standard deviation logic for CSW% & usage
    # ─────────────────────────────────────────────────────────────

    # 1) Exclude pitches with < 5% OVERALL usage from any consideration
    MIN_OVERALL_USAGE = 5.0  # percent of total pitches across all buckets

    overall_usage_cnt = usage_cnt.sum(axis=0)  # per pitch type (across all buckets)
    total_pitches = overall_usage_cnt.sum()
    overall_usage_pct = overall_usage_cnt / total_pitches * 100.0

    low_overall = overall_usage_pct < MIN_OVERALL_USAGE  # Series indexed by PitchType

    # Stats copies: low-overall pitches -> NaN so they don't affect medians/SDs
    csw_stats = csw_pct_b.copy().astype(float)
    usage_stats = usage_pct.copy().astype(float)
    csw_stats.loc[:, low_overall] = np.nan
    usage_stats.loc[:, low_overall] = np.nan

    # 2) Row-wise medians and standard deviations for CSW%
    row_median_csw = csw_stats.median(axis=1, skipna=True)
    row_std_csw = csw_stats.std(axis=1, ddof=0, skipna=True).replace(0, np.nan)

    z_csw = csw_stats.sub(row_median_csw, axis=0).div(row_std_csw, axis=0)

    # 3) Row-wise medians and standard deviations for usage%
    row_median_usage = usage_stats.median(axis=1, skipna=True)
    row_std_usage = usage_stats.std(axis=1, ddof=0, skipna=True).replace(0, np.nan)

    z_usage = usage_stats.sub(row_median_usage, axis=0).div(row_std_usage, axis=0)

    # 4) Build metric from z-scores only (no arbitrary thresholds)
    #    good_csw  = CSW above row median
    #    bad_csw   = CSW below row median
    #    under_use = usage below row median
    #    over_use  = usage above row median
    good_csw = z_csw.clip(lower=0)          # >0 where CSW is high vs row median
    bad_csw = (-z_csw).clip(lower=0)        # >0 where CSW is low
    under_use = (-z_usage).clip(lower=0)    # >0 where usage is low
    over_use = z_usage.clip(lower=0)        # >0 where usage is high

    # Positive (green): high CSW & low usage
    # Negative (red):   low CSW  & high usage
    metric = good_csw * under_use - bad_csw * over_use

    # Anything NaN or excluded by overall-usage rule -> neutral (0)
    metric = metric.fillna(0.0)
    metric.loc[:, low_overall] = 0.0

    # Symmetric color range around 0 so 0 = white, ±max = full red/green
    with np.errstate(all="ignore"):
        max_abs = float(np.nanmax(np.abs(metric.values))) if metric.values.size else 0.0
    if not np.isfinite(max_abs) or max_abs == 0.0:
        zmin, zmax = -1.0, 1.0
    else:
        zmin, zmax = -max_abs, max_abs

    csw_bucket_fig = px.imshow(
        metric,
        x=csw_pct_b.columns.tolist(),
        y=csw_pct_b.index.tolist(),
        color_continuous_scale=THEME["div"],      # red–white–green
        color_continuous_midpoint=0,
        zmin=zmin,
        zmax=zmax,
        aspect="auto",
        height=BUCKET_HEATMAP_H,
    )

    # Labels stay as "CSW% (CSW events)" – same as before
    csw_bucket_fig.update_traces(
        text=annotate(csw_pct_b, csw_succ_b),
        texttemplate="%{text}",
        textfont_size=12,
    )
    csw_bucket_fig.update_layout(
        title=dict(
            text=(
                "CSW % by Bucket (green = high CSW & under-used; red = low CSW & over-used)<br>"
                "<span style='font-size:12px;'>Color = product of CSW and usage z-scores; pitches <5% overall usage are shown neutral</span>"
            )
        ),
        margin=dict(l=0, r=0, t=55, b=0),
    )
    csw_bucket_fig.update_yaxes(autorange="reversed", ticklabelposition="outside left")

    # ----------------------------------------------------------------
    # 3)  Avg velo heatmap (bucket × pitch) – column-normalized
    # ----------------------------------------------------------------
    velo_val = (
        df_exp.groupby(["BucketList", "PitchType"])["Velo"]
              .mean()
              .unstack()
              .reindex(index=ROW_ORDER, columns=usage_cnt.columns)
    )
    if velo_val.dropna(how="all").empty:
        blank = go.Figure().update_layout(
            title="Avg Velo by Bucket (no data for current filter)",
            xaxis_visible=False, yaxis_visible=False)
        return freq_fig, csw_fig, usage_fig, csw_bucket_fig, blank

    # annotations (value + (n))
    txt = velo_val.copy().round(1).astype(str)
    for r in velo_val.index:
        for c in velo_val.columns:
            n = int(usage_cnt.loc[r, c])
            txt.loc[r, c] = f"{velo_val.loc[r, c]:.1f}<br>({n})" if n else ""

    # normalize columns 0–1 to give each pitch its own color scale
    norm = velo_val.copy()
    for col in norm.columns:
        col_min, col_max = norm[col].min(), norm[col].max()
        if pd.isna(col_min) or pd.isna(col_max) or col_max == col_min:
            norm[col] = 0.5
        else:
            norm[col] = (norm[col] - col_min) / (col_max - col_min)

    velo_fig = px.imshow(
        norm,
        x=velo_val.columns.tolist(),
        y=velo_val.index.tolist(),
        color_continuous_scale=THEME["neutral"],
        aspect="auto", height=BUCKET_HEATMAP_H,
        labels={"x": "Pitch Type", "y": "Bucket", "color": "Rel. Velo"}
    )
    velo_fig.update_traces(text=txt.values, texttemplate="%{text}", textfont_size=12)
    velo_fig.update_coloraxes(showscale=False)
    velo_fig.update_layout(title="Avg Velo by Bucket",
                           margin=dict(l=30, r=10, t=40, b=30))
    velo_fig.update_yaxes(autorange="reversed", ticklabelposition="outside left")
    velo_fig.update_xaxes(side="bottom", tickangle=45)

    return freq_fig, csw_fig, usage_fig, csw_bucket_fig, velo_fig

# ------------------------------------------------------------------
#  PUT-AWAY SCENARIOS  (Kill%, PATS, etc.)
# ------------------------------------------------------------------
@app.callback(
    Output("putaway-table-left", "data"),
    Output("putaway-table-left", "tooltip_data"),
    Input("pitcher-dd", "value"),
    Input("date-dd",    "value"),
    Input("bat-hand-dd","value"),
    State("raw-data",   "data"),
)
def update_putaway_table(pitcher, dates, hand, raw_json):
    if not pitcher or not raw_json:
        return [], []

    df = pd.read_json(raw_json, orient="split")
    df["GameDate"] = pd.to_datetime(df["GameDate"]).dt.date

    # filters
    df_p = df[df["Pitcher"] == pitcher].copy()
    if dates:
        sel_dates = [pd.to_datetime(d).date() for d in dates]
        df_p = df_p[df_p["GameDate"].isin(sel_dates)]
    if hand != "All":
        df_p = df_p[df_p["BatterHand"] == hand]

    # metrics
    P = pd.Series(putaway_metrics(df_p))
    T = pd.Series(putaway_metrics(df))

    rows = [
        dict(Metric="Kill %",            P=f"{P.kill:.1f} %",  T=f"{T.kill:.1f} %",  Diff=f"{P.kill-T.kill:+.1f} %",  Diff_num=P.kill-T.kill),
        dict(Metric="PATS",              P=f"{P.pats:.2f}",    T=f"{T.pats:.2f}",    Diff=f"{P.pats-T.pats:+.2f}",    Diff_num=P.pats-T.pats),
        dict(Metric="Avg Fouls 2-Str",   P=f"{P.foul_avg:.2f}",T=f"{T.foul_avg:.2f}",Diff=f"{P.foul_avg-T.foul_avg:+.2f}", Diff_num=P.foul_avg-T.foul_avg),
        dict(Metric="Avg Balls 2-Str",   P=f"{P.ball_avg:.2f}",T=f"{T.ball_avg:.2f}",Diff=f"{P.ball_avg-T.ball_avg:+.2f}", Diff_num=P.ball_avg-T.ball_avg),
        dict(Metric="% IP-No-Out 2-Str", P=f"{P.ipno_pct:.1f} %", T=f"{T.ipno_pct:.1f} %", Diff=f"{P.ipno_pct-T.ipno_pct:+.1f} %", Diff_num=P.ipno_pct-T.ipno_pct),
    ]

    explainer = {
        "Kill %"           : "%% of plate appearances that reach **two strikes** and end on the **next pitch** with a strikeout.",
        "PATS"             : "Average **Pitches After Two-Strike** per two-strike PA.",
        "Avg Fouls 2-Str"  : "Average **foul balls after first two-strike pitch** per two-strike PA.",
        "Avg Balls 2-Str"  : "Average **balls after first two-strike pitch** per two-strike PA.",
        "% IP-No-Out 2-Str": "Share of **two-strike pitches** that become *In-Play, No-Out* (per-pitch metric).",
    }
    tips = [{
        "Metric": {"value": explainer[r["Metric"]], "type": "markdown"},
        "P":      {"value": f"Pitcher’s {r['Metric']}", "type": "markdown"},
        "T":      {"value": "Team/league reference", "type": "markdown"},
        "Diff":   {"value": "Pitcher minus reference", "type": "markdown"},
    } for r in rows]

    return rows, tips


# ────────────────────────────────────────────────────────────────────
#  MIXED-STACK + ROLLING CSW
# ────────────────────────────────────────────────────────────────────
@app.callback(
    Output("mix-csw", "figure"),
    Input("filtered-data", "data"),
    Input("count-filter",     "value"),
)
def update_mix_csw(sub_json, count_filter):
    if not sub_json:
        return go.Figure()

    # load and prep
    df = pd.read_json(sub_json, orient="split")
    df["GameDate"] = pd.to_datetime(df["GameDate"])

    # ── COUNT FILTER ───────────────────────────────────────────────
    if count_filter == "Early":
        df = df[df["Count_pre"].fillna(df["Count"]).isin(EARLY_CT)]
    elif count_filter == "Late":
        df = df[df["Count"].isin(LATE_CT)]

    csw_set = {"Swinging Strike","Called Strike","Foul Tip","Swinging Strike (Blocked)"}
    df["csw"] = df["PitchResult"].isin(csw_set).astype(int)

    # 1) per-game mix %
    usage_pct = (
        df.groupby(["GameDate","PitchType"]).size()
          .groupby(level=0, group_keys=False)
          .apply(lambda s: 100*s/s.sum())
          .unstack(fill_value=0)
    )

    # 2) daily CSW & 7-day rolling
    csw_daily = df.groupby("GameDate")["csw"].mean().mul(100)
    csw7 = csw_daily.rolling(window=7, min_periods=1).mean()

    # scale bars to match CSW7
    usage_scaled = usage_pct.div(100).mul(csw7, axis=0)

    # build figure
    fig = go.Figure()

    for pt in usage_scaled.columns:
        fig.add_trace(go.Bar(
            x=usage_scaled.index,
            y=usage_scaled[pt],
            name=pt,
            marker_line_width=0
        ))

    # overlay the black line (no markers)
    fig.add_trace(go.Scatter(
        x=csw7.index, y=csw7.values,
        mode="lines",
        line=dict(color="black", width=2.5),
        name="7-Day CSW%"
    ))

    fig.update_layout(
        title="Daily Pitch-Mix Scaled to 7-Day CSW%",
        barmode="stack",
        xaxis=dict(title="", tickformat="%b %d"),
        yaxis=dict(title="CSW% (bars sum to CSW7)", range=[0,100]),
        legend=dict(orientation="h", y=-0.2, x=0.5,xanchor="center",font=dict(size=10)),
        margin=dict(l=40, r=20, t=50, b=30),
        hovermode="x unified"
    )
    return fig

# ── CUSTOM PLOT ───────────────────────────────────────────────


# ------------------------------------------------------------------
#  CUSTOM OPTIONS
# ------------------------------------------------------------------


@app.callback(
    Output("cust-count", "options"),
    Output("cust-ptype", "options"),
    Input("filtered-data", "data")
)
def set_custom_options(sub_json):
    if not sub_json:
        return [], []
    df = pd.read_json(sub_json, orient="split")
    cnt = sorted(df["Count"].unique())
    pt  = sorted(df["PitchType"].unique())
    return (
        [{"label":c, "value":c} for c in cnt],
        [{"label":p, "value":p} for p in pt],
    )

# ------------------------------------------------------------------
#  USAGE-LINE
# ------------------------------------------------------------------
@app.callback(
    Output("usage-line", "figure"),
    Input("filtered-data", "data"),
    Input("ptype-dd","value"),
    Input("count-filter",    "value"),
)

def usage_line(sub_json, ptype, count_filter):
    if not sub_json:
        return go.Figure()

    df = pd.read_json(sub_json, orient="split")

    # ── COUNT FILTER ───────────────────────────────────────────────
    if count_filter == "Early":
        df = df[df["Count_pre"].fillna(df["Count"]).isin(EARLY_CT)]
    elif count_filter == "Late":
        df = df[df["Count"].isin(LATE_CT)]


    # ⬇️  keep only the selected pitch-type if one was chosen
    if ptype:
        p_clean = ptype.strip().upper()
        df = df[df["PitchType"].str.strip().str.upper() == p_clean]
    if df.empty:
        fig = go.Figure()
        fig.update_layout(title="No pitches for selected filters")
        return fig

    usage = (
        df.groupby(["GameDate","PitchType"])
          .size()
          .groupby(level=0,group_keys=False)
          .apply(lambda s:100*s/s.sum())
          .reset_index(name="Usage")
          .sort_values("GameDate")
    )
    df["csw"] = df["PitchResult"].isin(
        {"Swinging Strike","Called Strike","Foul Tip","Swinging Strike (Blocked)"}
    )
    csw_daily = (
        df.groupby("GameDate")["csw"]
          .mean().mul(100)
          .reset_index(name="CSW")
          .sort_values("GameDate")
    )

    usage["GameDate"]     = pd.to_datetime(usage["GameDate"])
    csw_daily["GameDate"] = pd.to_datetime(csw_daily["GameDate"])

    fig = go.Figure()
    for pt,grp in usage.groupby("PitchType"):
        fig.add_trace(go.Scatter(
            x=grp["GameDate"], y=grp["Usage"],
            mode="lines+markers",
            name=f"{pt} Usage %",
            hovertemplate="%{y:.1f}%<extra>"+pt+"</extra>"
        ))
    fig.add_trace(go.Bar(
        x=csw_daily["GameDate"], y=csw_daily["CSW"],
        name="CSW %",
        opacity=0.35,
        marker_color="grey",
        yaxis="y2",
        hovertemplate="CSW %: %{y:.1f}<extra></extra>"
    ))
    fig.update_layout(
        title="Pitch-mix vs Daily CSW %",
        xaxis=dict(title="",tickformat="%b %d"),
        yaxis=dict(title="Usage %",range=[0,100]),
        yaxis2=dict(title="CSW %",overlaying="y",side="right",range=[0,100],showgrid=False),
        legend=dict(orientation="h",yanchor="bottom",y=1.02,xanchor="right",x=1),
        margin=dict(l=20,r=20,t=40,b=20)
    )
    return fig


# ──────────────────────────────────────────────────────────────
# MINI COUNT KPI  (RV tiles – colour-graded & league delta)
# ──────────────────────────────────────────────────────────────
EARLY_CT    = {f"{b}-{s}" for b in range(4) for s in (0, 1)}
LATE_CT     = {f"{b}-2"  for b in range(4)}
HITTER_CT   = {"1-0","2-0","2-1","3-0","3-1"}
PITCHER_CT  = {"0-1","0-2","1-2"}
EVEN_CT     = {"0-0","1-1","2-2"}

# ─────────  colour helpers for RV  ──────────
TILE_CLIP = THEME["rv_clip"]


def colour_for_rv(v_raw, cnt, *, scale="delta", clip=TILE_CLIP):
    base = LG_BY_COUNT.get(cnt, 0.0) if cnt else 0.0
    v = v_raw - base if scale == "delta" else v_raw
    v = float(np.clip(v, -clip, clip))
    # map [-clip, clip] → RdBu
    lo, hi = -clip, clip
    # slightly translucent tile so text pops
    return div_colour(v, lo, hi, alpha=0.95)

@app.callback(
    Output("count-summary", "children"),
    Input("filtered-data", "data"),
    State("raw-data",       "data"))
def update_count_kpi(sub_json, raw_json):
    if not sub_json:
        return []

    df_team = pd.read_json(sub_json, orient="split")
    df_lg   = pd.read_json(raw_json,  orient="split")

    def avg_rv(df, bucket):
        s = df[df["Count"].isin(bucket)]["RunValue"]
        return float(s.mean()) if len(s) else np.nan

    rv_hit_team   = avg_rv(df_team, HITTER_CT)
    rv_pitch_team = avg_rv(df_team, PITCHER_CT)
    rv_early_team = avg_rv(df_team, EARLY_CT)
    rv_late_team  = avg_rv(df_team, LATE_CT)

    rv_hit_lg   = avg_rv(df_lg,  HITTER_CT)
    rv_pitch_lg = avg_rv(df_lg,  PITCHER_CT)
    rv_early_lg = avg_rv(df_lg,  EARLY_CT)
    rv_late_lg  = avg_rv(df_lg,  LATE_CT)

    def fmt(v):
        return "n/a" if (v is None or pd.isna(v)) else f"{v:+.3f}"

    def tile(label, team_val, lg_val):
        # delta to league (colour driver)
        diff = (team_val - lg_val) if (pd.notna(team_val) and pd.notna(lg_val)) else np.nan

        # Always color by delta (league relative). Flip so “more negative RV” (good) → greener.
        v_for_colour = 0.0 if pd.isna(diff) else float(np.clip(diff, -TILE_CLIP, TILE_CLIP))
        score = -v_for_colour
        bg    = good_bad_bg(score, clip=TILE_CLIP, alpha=0.95)
        fg    = text_color_for(bg)

        # Hover help (single attribute → no extra deps). \n renders as newlines in most browsers.
        help_txt = (
            f"{label}\n"
            f"Number = Avg Run Value (per pitch).\n"
            f" • Negative favors the PITCHER; positive favors the HITTER.\n"
            f"Color = league-relative.\n"
            f" • Green = better than league (more negative RV).\n"
            f" • Red = worse than league (less negative / more positive).\n"
            f"Pitcher Avg RV: {fmt(team_val)}\n"
            f"League Avg RV:  {fmt(lg_val)}\n"
            f"Δ vs League:     {fmt(diff)}"
        )

        return html.Div([
            html.Div(label,                style={"fontSize":12,"color":fg,"marginBottom":2}),
            html.Div(fmt(team_val),        style={"fontWeight":700,"fontSize":22,"color":fg}),
            html.Div(f"{fmt(diff)} vs Lg", style={"fontSize":11,"color":fg})
            # If you decide you want the league number visible (not just on hover), uncomment:
            # , html.Div(f"Lg: {fmt(lg_val)}", style={"fontSize":11,"color":fg})
        ], style={
            "background": bg, "borderRadius": 10, "padding": "8px 16px",
            "minWidth": 160, "textAlign": "center", "cursor": "help"
        }, title=help_txt)

    return [
        tile("Avg RV – Hitter Cts",   rv_hit_team,   rv_hit_lg),
        tile("Avg RV – Pitcher Cts",  rv_pitch_team, rv_pitch_lg),
        tile("Avg RV – Early Cts",    rv_early_team, rv_early_lg),
        tile("Avg RV – Late Cts",     rv_late_team,  rv_late_lg),
    ]

# ──────────────────────────────────────────────────────────────
# NAN WARNING
# ──────────────────────────────────────────────────────────────

@app.callback(
    Output("nan-warning", "children"),
    Input("filtered-data", "data"),
)
def update_nan_warning(sub_json):
    if not sub_json:
        return ""

    df = pd.read_json(sub_json, orient="split")
    nan_n, total_n, pct = nan_pitchtype_stats(df)

    if total_n == 0 or nan_n == 0:
        return ""

    return html.Div([
        html.Span("⚠", style={"fontSize": "16px"}),
        html.Span(
            f"{nan_n:,} untagged (NaN) pitches excluded from pitch-type tables/heatmaps "
            f"({pct:.1f}% of {total_n:,} total pitches)."
        )
    ])


# ──────────────────────────────────────────────────────────────
# PITCH-TYPE RV TILES (Total RV headline; colored by league-relative Avg RV)
# ──────────────────────────────────────────────────────────────
@app.callback(
    Output("pitch-rv-tiles", "children"),
    Input("raw-data", "data"),
    Input("pitcher-dd", "value"),
    Input("date-dd", "value"),
)
def update_pitch_rv_tiles(raw_json, pitcher, date_list):
    if not raw_json or not pitcher:
        return []

    raw = pd.read_json(raw_json, orient="split").copy()
    if raw.empty:
        return []

    # Normalize pitch-type strings WITHOUT turning NaN into "nan"
    raw["PitchType"] = raw["PitchType"].astype("string").str.strip()

    # Treat common "missing" tokens as missing
    missing_tokens = {"", "nan", "none", "na", "n/a"}
    raw.loc[raw["PitchType"].str.lower().isin(missing_tokens), "PitchType"] = pd.NA

    # League baseline: Avg RV per pitch type (exclude unknown PitchType)
    lg_by_pt = (
        raw[raw["PitchType"].notna()]
          .groupby("PitchType")["RunValue"].mean()
          .to_dict()
    )

    # Pitcher subset (respect pitcher + date filters, ignore batter-hand dropdown on purpose)
    df = raw[raw["Pitcher"] == pitcher].copy()
    if date_list:
        if isinstance(date_list, str):
            date_list = [date_list]
        date_list = [str(d) for d in date_list]
        df["_GameDateStr"] = df["GameDate"].astype(str)
        df = df[df["_GameDateStr"].isin(date_list)]

    if df.empty:
        return []

    # Only use pitches with known pitch type for pitch-type usage logic
    df_pt = df[df["PitchType"].notna()].copy()
    if df_pt.empty:
        return []

    total_n = len(df_pt)

    # Hand totals for usage-vs-hand denominators (known pitch types only)
    bh = df_pt["BatterHand"].astype(str).str.upper().str.strip()
    is_r = bh.str.startswith("R")
    is_l = bh.str.startswith("L")
    total_r = int(is_r.sum())
    total_l = int(is_l.sum())


    # Summaries by pitch type
    g = (
        df_pt.groupby("PitchType", as_index=False)
            .agg(
                N=("RunValue", "size"),
                RV_sum=("RunValue", "sum"),
                RV_avg=("RunValue", "mean"),
            )
    )
    g["usage"] = g["N"] / total_n

    # 3% usage cutoff (overall usage for this pitcher+date selection)
    g = g[g["usage"] >= MIN_PITCHTYPE_USAGE_TILE].copy()
    if g.empty:
        return []

    # Split usage by hand
    df_pt["_BH"] = bh.str[0]  # 'R'/'L'/etc
    pivot = df_pt.pivot_table(
        index="PitchType", columns="_BH", values="RunValue",
        aggfunc="size", fill_value=0
    )

    def get_pivot_count(pt, col):
        if pivot is None:
            return 0
        if pt not in pivot.index or col not in pivot.columns:
            return 0
        return int(pivot.loc[pt, col])

    g["N_R"] = g["PitchType"].map(lambda pt: get_pivot_count(pt, "R"))
    g["N_L"] = g["PitchType"].map(lambda pt: get_pivot_count(pt, "L"))
    g["use_R"] = g["N_R"] / total_r if total_r else 0.0
    g["use_L"] = g["N_L"] / total_l if total_l else 0.0

    # League-relative coloring driver (Avg RV delta vs league for that pitch type)
    g["LG_avg"] = g["PitchType"].map(lg_by_pt)
    g["diff"] = g["RV_avg"] - g["LG_avg"]

    # Sort tiles: most-used first (you can switch to RV_sum if you want chaos)
    g = g.sort_values(["usage", "N"], ascending=[False, False])

    def fmt_rv_sum(v):
        return "n/a" if (v is None or pd.isna(v)) else f"{float(v):+.2f}"  # total RV (hover)

    def fmt_rv_avg(v):
        return "n/a" if (v is None or pd.isna(v)) else f"{float(v):+.3f}"  # avg RV per pitch (tile headline)


    def fmt_pct(v):
        return "—" if (v is None or pd.isna(v)) else f"{100*float(v):.1f}%"

    def fmt_int(v):
        return "0" if (v is None or pd.isna(v)) else f"{int(v):,}"

    tiles = []
    for _, r in g.iterrows():
        pt = str(r["PitchType"]).strip()
        label = pitch_abbr(pt)

        team_avg = r["RV_avg"]
        lg_avg = r["LG_avg"]
        diff = r["diff"]

        # Same color logic as count tiles: clamp delta, flip sign so more-negative (better) => greener
        v_for_colour = 0.0 if pd.isna(diff) else float(np.clip(diff, -TILE_CLIP, TILE_CLIP))
        score = -v_for_colour
        bg = good_bad_bg(score, clip=TILE_CLIP, alpha=0.95)
        fg = text_color_for(bg)

        lg_avg = r["LG_avg"]
        diff   = r["diff"]

        help_txt = (
            f"{label} ({pt})\n"
            f"Total RV: {fmt_rv_sum(r['RV_sum'])}\n"
            f"League Avg RV: {('n/a' if pd.isna(lg_avg) else f'{float(lg_avg):+.3f}')}\n"
            f"Δ Avg vs Lg: {('n/a' if pd.isna(diff) else f'{float(diff):+.3f}')}\n"
            f"N: {int(r['N'])}\n"
            f"Usage overall: {fmt_pct(r['usage'])}\n"
            f"Usage vs RHB: {fmt_pct(r['use_R'])} (N={int(r['N_R'])})\n"
            f"Usage vs LHB: {fmt_pct(r['use_L'])} (N={int(r['N_L'])})"
        )


        tiles.append(
            html.Div(
                children=[
                    html.Div(label, style={"fontSize": 12, "color": fg, "marginBottom": 2}),
                    html.Div(fmt_rv_avg(r["RV_avg"]), style={"fontWeight": 800, "fontSize": 22, "color": fg}),
                    html.Div(f"Usage {fmt_pct(r['usage'])}", style={"fontSize": 11, "color": fg}),
                    html.Div(
                        f"N {fmt_int(r['N'])} · RHB {fmt_pct(r['use_R'])} · LHB {fmt_pct(r['use_L'])}",
                        style={"fontSize": 11, "color": fg},
                    ),
                ],
                style={
                    "background": bg,
                    "borderRadius": 10,
                    "padding": "8px 16px",
                    "minWidth": 170,
                    "textAlign": "center",
                    "cursor": "help",
                },
                title=help_txt,
            )
        )

    return tiles


# ───────────────────────────────────────────────────────────────
# MIX / REPEAT TABLE  (replaces old SSR bar)
# ───────────────────────────────────────────────────────────────
@app.callback(
    Output("mix-table","data"),
    Output("mix-table","tooltip_data"),      # ← NEW
    Input("filtered-data","data"))

def update_mix_table(sub_json):
    if not sub_json:
        return [], []

    df = pd.read_json(sub_json, orient="split")

    # build prev-count per pitch using true AB order
    chunks = []
    for _, ab in _group_by_ab(df):
        if "PITCH_NUMBER_AB" in ab.columns and ab["PITCH_NUMBER_AB"].notna().any():
            ab = ab.sort_values("PITCH_NUMBER_AB")
        else:
            ab = ab.sort_values("DateTime")
        ab = ab.copy()
        ab["prev_cnt"] = ab["Count"].shift()
        chunks.append(ab)
    if not chunks:
        return [], []
    df = pd.concat(chunks, ignore_index=True)

    masks = {
        "Overall"  : np.ones(len(df), bool),
        "Early"    : df["prev_cnt"].isin(EARLY_CT),
        "Late"     : df["prev_cnt"].isin(LATE_CT),
        "Hitter's" : df["prev_cnt"].isin(HITTER_CT),
        "Pitcher's": df["prev_cnt"].isin(PITCHER_CT),
    }

    rows, tips = [], []
    for name, m in masks.items():
        met = mix_metrics(df, m)
        rows.append(dict(
            Bucket    = name,
            MixPct    = f"{100-met['repeat_pct']:.1f}%" if pd.notna(met['repeat_pct']) else "n/a",
            RepeatPct = f"{met['repeat_pct']:.1f}%"     if pd.notna(met['repeat_pct']) else "n/a",
            BlendPct  = f"{met['blend_pct']:.1f}%"      if pd.notna(met['blend_pct']) else "n/a",
        ))
        tips.append({
            "Bucket"   : {"value":f"{name} pitch-count bucket","type":"markdown"},
            "MixPct"   : {"value":"How often we change pitch type from one pitch to the next","type":"markdown"},
            "RepeatPct": {"value":"How often we throw the **same** pitch back-to-back","type":"markdown"},
            "BlendPct" : {"value":"Even-Mix % – 0 = one pitch dominates, 100 = perfectly even usage","type":"markdown"},
        })
    return rows, tips


# ───────────────────────────────────────────────────────────────
# GET-ME-RIGHT TABLE
# ───────────────────────────────────────────────────────────────
@app.callback(
    Output("gmr-table","data"),
    Output("gmr-table","tooltip_data"),
    Input("filtered-data","data"))
def update_gmr_table(sub_json):
    if not sub_json:
        return [], []

    df = pd.read_json(sub_json, orient="split")
    tbl = get_me_right(df).head(25)

    # Save full names for tooltips, shrink display to abbreviations
    tbl["Prev_full"]   = tbl["PrevPitch"].astype(str)
    tbl["Helper_full"] = tbl["HelperPitch"].astype(str)
    tbl["PrevPitch"]   = tbl["Prev_full"].map(pitch_abbr)
    tbl["HelperPitch"] = tbl["Helper_full"].map(pitch_abbr)

    # Format %
    tbl["SuccessRate"] = (tbl["SuccessRate"]*100).map("{:.1f}%".format)

    # Per-row tooltips with full names
    tips = []
    for _, r in tbl.iterrows():
        tips.append({
            "PrevPitch":  {"value": f"{r['Prev_full']}",   "type": "text"},
            "HelperPitch":{"value": f"{r['Helper_full']}", "type": "text"},
            "Success":    {"value": "Successful A→B→A sequences (CSW on the return A)", "type": "markdown"},
            "Attempts":   {"value": "Total A→B→A attempts", "type": "markdown"},
            "SuccessRate":{"value": "Success ÷ Attempts",   "type": "markdown"},
        })

    # Only return the visible columns
    out = tbl[["PrevPitch","HelperPitch","Success","Attempts","SuccessRate"]]
    return out.to_dict("records"), tips


# ───────────────────────────────────────────────────────────────
# REPEAT vs MIX  (dumbbell chart)
# ───────────────────────────────────────────────────────────────
@app.callback(Output("repeat-mix","figure"),
              Input("filtered-data","data"))
def update_repeat_mix(sub_json):
    if not sub_json:
        return go.Figure()

    df = pd.read_json(sub_json, orient="split")
    rows = []
    for _, ab in _group_by_ab(df):
        ab = ab.sort_values("PITCH_NUMBER_AB") if (
            "PITCH_NUMBER_AB" in ab.columns and ab["PITCH_NUMBER_AB"].notna().any()
        ) else ab.sort_values("DateTime")
        ab["prev_pt"] = ab["PitchType"].shift()
        ab = ab.dropna(subset=["prev_pt"])
        if ab.empty:
            continue
        ab["variant"] = np.where(ab["PitchType"] == ab["prev_pt"], "Repeat", "Mix")
        ab["is_csw"]  = ab["PitchResult"].isin(CSW_SET).astype(int)
        rows.append(ab[["prev_pt","variant","is_csw"]])

    if not rows:
        return go.Figure().update_layout(title="No transitions")

    d = pd.concat(rows, ignore_index=True)

    # require at least 5 transitions from a given first pitch
    good_first = d.groupby("prev_pt").size().ge(5)
    d = d[d["prev_pt"].isin(good_first[good_first].index)]
    if d.empty:
        return go.Figure().update_layout(title="Too few transitions")

    # mean CSW% and counts by variant
    stats  = d.groupby(["prev_pt","variant"])["is_csw"].mean().unstack() * 100
    counts = d.groupby(["prev_pt","variant"]).size().unstack(fill_value=0)

    # ensure both columns exist
    stats  = stats.reindex(columns=["Mix","Repeat"])
    counts = counts.reindex(columns=["Mix","Repeat"], fill_value=0)

    # sort by effect size (Repeat − Mix), most interesting first
    order = (stats["Repeat"] - stats["Mix"]).sort_values(ascending=False).index
    stats  = stats.loc[order]
    counts = counts.loc[order]

    fig = go.Figure()
    for i, (first_pitch, row) in enumerate(stats.iterrows()):
        mix = row.get("Mix");    rep = row.get("Repeat")
        mix_n = counts.loc[first_pitch, "Mix"]
        rep_n = counts.loc[first_pitch, "Repeat"]
        if pd.notna(mix):
            fig.add_trace(go.Scatter(
                x=[mix], y=[i], mode="markers",
                marker=dict(size=10),  # colors inherit theme
                name="Mix", legendgroup="Mix", showlegend=(i == 0),
                hovertemplate=f"{first_pitch}<br>Mix CSW %: %{{x:.1f}} ({mix_n})<extra></extra>"
            ))
        if pd.notna(rep):
            fig.add_trace(go.Scatter(
                x=[rep], y=[i], mode="markers",
                marker=dict(size=10),
                name="Repeat", legendgroup="Repeat", showlegend=(i == 0),
                hovertemplate=f"{first_pitch}<br>Repeat CSW %: %{{x:.1f}} ({rep_n})<extra></extra>"
            ))
        if pd.notna(mix) and pd.notna(rep):
            fig.add_trace(go.Scatter(
                x=[mix, rep], y=[i, i], mode="lines",
                line=dict(width=2), hoverinfo="skip"
            ))

    fig.update_layout(
        title="Doubledown vs Mix – CSW % on the 2nd pitch after ‘first’",
        xaxis=dict(title="CSW %", range=[0, 100]),
        yaxis=dict(
            title="First pitch of the pair",
            tickmode="array",
            tickvals=list(range(len(stats))),
            ticktext=list(stats.index),
            automargin=True
        ),
        showlegend=False,
        margin=dict(l=120, r=20, t=50, b=40)
    )
    return fig  # ← THE IMPORTANT PART


# ────────────────────────────────────────────────────────────────────
# 7-DAY ROLLING AVERAGES CHART  (fixed denominator for Usage %)
# ────────────────────────────────────────────────────────────────────
@app.callback(
    Output("rolling-graph", "figure"),
    Input("filtered-data", "data"),
    Input("rolling-metric", "value"),
    Input("cust-count",     "value"),
    Input("cust-ptype",     "value"),
    Input("count-filter",   "value"),
    Input("rolling-overall-csw", "value"),   # 👈 NEW
)
def build_rolling(sub_json, metric, cnt_filter, pt_filter, count_filter, overlay_flags):
    if not sub_json:
        return go.Figure().update_layout(title="Select a pitcher")

    df = pd.read_json(sub_json, orient="split")

    # ── COUNT FILTER ───────────────────────────────────────────────
    if count_filter == "Early":
        df = df[df["Count_pre"].fillna(df["Count"]).isin(EARLY_CT)]
    elif count_filter == "Late":
        df = df[df["Count"].isin(LATE_CT)]

    if cnt_filter:
        df = df[df["Count"].isin(cnt_filter)]
    if df.empty:
        return go.Figure().update_layout(title="No data for current filters")

    df["GameDate"] = pd.to_datetime(df["GameDate"]).dt.normalize()
    df = df.sort_values("GameDate")

    # We'll compute an "overall" CSW7 on the subset currently in play.
    csw_set = {"Swinging Strike","Called Strike","Foul Tip","Swinging Strike (Blocked)"}

    # ----------------------------------------------------------------
    # ROLLING USAGE – keep denominator across all pitch types; filter
    # visible lines by pt_filter but **overall CSW7** respects pt_filter.
    # ----------------------------------------------------------------
    if metric == "RollingUsage":
        daily_cnt  = df.groupby(["GameDate", "PitchType"]).size()
        usage_pct  = (daily_cnt
                      .groupby(level=0, group_keys=False)
                      .apply(lambda s: 100 * s / s.sum()))
        wide       = usage_pct.unstack(fill_value=0).sort_index()
        roll7      = wide.rolling(7, min_periods=1).mean()

        # Hide unselected lines but keep the math intact.
        if pt_filter:
            keep_cols = [c for c in roll7.columns if c in pt_filter]
            if not keep_cols:
                return go.Figure().update_layout(title="No matching pitch types")
            roll7 = roll7[keep_cols]

        roll = (roll7.stack()
                     .reset_index()
                     .rename(columns={0: "value"}))

        ylab = "Usage % (7-day)"
        ttl  = "7-Day Rolling Usage %"

    # ----------------------------------------------------------------
    # ROLLING VELO / ROLLING CSW – these aggregate only selected pitch types.
    # ----------------------------------------------------------------
    else:
        if pt_filter:
            df = df[df["PitchType"].isin(pt_filter)]
            if df.empty:
                return go.Figure().update_layout(title="No data for current filters")

        df["is_csw"] = df["PitchResult"].isin(csw_set).astype(int)
        daily_velo = df.groupby(["GameDate", "PitchType"])["Velo"].mean()
        daily_csw  = df.groupby(["GameDate", "PitchType"])["is_csw"].mean() * 100

        if metric == "RollingVelo":
            series = daily_velo
            ylab, ttl = "Avg Velo (7-day)", "7-Day Rolling Velo"
        else:  # "RollingCSW"
            series = daily_csw
            ylab, ttl = "CSW % (7-day)", "7-Day Rolling CSW %"

        wide = series.unstack(fill_value=np.nan).sort_index()
        roll7 = wide.rolling(7, min_periods=1).mean()

        roll = (roll7.stack()
                       .reset_index()
                       .rename(columns={0: "value"}))

    # ---------- plot the main series ----------
    fig = go.Figure()
    for pt, grp in roll.groupby("PitchType") if "PitchType" in roll.columns else [("All", roll)]:
        fig.add_trace(go.Scatter(
            x=grp["GameDate"], y=grp["value"],
            mode="lines+markers", name=(pt if pt != "All" else metric)
        ))

    # ---------- optional overlay: overall 7-day CSW% ----------
    if overlay_flags and "csw7" in overlay_flags:
        df_csw = df.copy()
        if metric == "RollingUsage" and pt_filter:
            df_csw = df_csw[df_csw["PitchType"].isin(pt_filter)]
        if not df_csw.empty:
            df_csw["is_csw"] = df_csw["PitchResult"].isin(csw_set).astype(int)
            csw_daily_overall = df_csw.groupby("GameDate")["is_csw"].mean().mul(100)
            csw7_overall = csw_daily_overall.rolling(7, min_periods=1).mean()
            fig.add_trace(go.Scatter(
                x=csw7_overall.index, y=csw7_overall.values,
                mode="lines", name="Overall CSW% (7-day)",
                line=dict(color="black", width=3),
                hovertemplate="Overall CSW% (7d): %{y:.1f}<extra></extra>"
            ))


    fig.update_layout(
        title=ttl,
        xaxis_title="Date",
        yaxis_title=ylab,
        legend=dict(orientation="h", y=-0.25, x=0.5, xanchor="center"),
        margin=dict(l=60, r=20, t=60, b=80),
        hovermode="x unified"
    )
    return fig

# Populate the pitch-type filter for scatter
@app.callback(
    Output("scatter-ptype", "options"),
    Output("scatter-ptype", "value"),
    Input("filtered-data", "data")
)
def set_scatter_ptypes(sub_json):
    if not sub_json:
        return [], []
    df = pd.read_json(sub_json, orient="split")
    opts = [{"label": p, "value": p} for p in sorted(df["PitchType"].unique())]
    vals = [o["value"] for o in opts]
    return opts, vals

# ------------------------------------------------------------------
#  UNIVERSAL SCATTER  (avg metrics on Y, fully custom axes / colour)
# ------------------------------------------------------------------
@app.callback(
    Output("csw-scatter", "figure"),
    Input("filtered-data", "data"),
    Input("scatter-x", "value"),
    Input("scatter-y", "value"),
    Input("scatter-color", "value"),
    Input("scatter-ptype", "value"),
    Input("scatter-agg", "value"),
)

def update_custom_scatter(sub_json, x_var, y_var, color_var, ptype_filter, agg_mode):
    if not sub_json:
        return go.Figure().update_layout(title="Select a pitcher")

    df = pd.read_json(sub_json, orient="split")
    if "GameDate" in df.columns:
        df["GameDate"] = pd.to_datetime(df["GameDate"], errors="coerce")

    # optional pitch-type filter (ignore missing types automatically)
    if ptype_filter:
        df = df[df["PitchType"].isin(ptype_filter)]
        if df.empty:
            return go.Figure().update_layout(title="No data for current filters")

    # ========= ban COLORING by CSW% (still OK on axes) =========
    if color_var == "CSW_pct":
        color_var = None

    # add CSW% only if axes need it
    if (x_var == "CSW_pct") or (y_var == "CSW_pct"):
        df["CSW_pct"] = df["PitchResult"].isin(CSW_SET).astype(float) * 100

    # --- numeric coercion for axis vars (prevents avg-mode crashes)
    for c in [x_var, y_var, color_var]:
        if c and c in df.columns and c not in ("PitchType", "Count", "PitchResult", "PLAY_RESULT"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # --- prepare Y series + label
    if y_var == "RunValue":
        df["_Y"] = pd.to_numeric(df.get("RunValue"), errors="coerce")
        y_lab = "Run Value"
    elif y_var == "CSW_pct":
        df["_Y"] = pd.to_numeric(df.get("CSW_pct"), errors="coerce")
        y_lab = "CSW %"
    elif y_var == "GameDate":
        df["_Y"] = df.get("GameDate")
        y_lab = "Game Date"
    else:
        df["_Y"] = pd.to_numeric(df.get(y_var), errors="coerce") if y_var in df.columns else np.nan
        y_lab = y_var

    # --- special color overrides you requested
    color_col = color_var

    if color_var == "PitchType" and "PitchType" in df.columns:
        df["__pitch_class"] = _pitch_class(df["PitchType"])
        color_col = "__pitch_class"

    if color_var == "Count":
        df["__count_group"] = _count_group(df)
        color_col = "__count_group"

    # --- robust symmetric limits for RunValue
    def _rv_limits(series):
        vals = pd.to_numeric(series, errors="coerce").dropna()
        if vals.empty:
            return -1.0, 1.0
        q = np.nanpercentile(np.abs(vals), 90)
        q = float(np.clip(q, 0.4, 1.2))
        return -q, q

    RV_DIVERGE = [
        [0.00, "rgb(192,57,43)"],
        [0.48, "rgb(242,172,146)"],
        [0.50, "rgb(247,227,121)"],  # yellow midpoint (visible)
        [0.52, "rgb(173,219,154)"],
        [1.00, "rgb(39,174,96)"],
    ]

    NONWHITE_SEQ = px.colors.sequential.Viridis

    # Force RAW when coloring by RunValue
    use_raw = (agg_mode == "raw") or (color_var == "RunValue")

    # GameDate continuous numeric scale (no buckets)
    if "GameDate" in df.columns:
        df["__gameday_num"] = (df["GameDate"].astype("int64", errors="ignore") // 86_400_000_000_000)

    # categorical or not?
    is_cat_color = (color_col in ("__pitch_class", "__count_group")) or (color_col in ("PitchType", "Count"))

    if use_raw:
        data = df.copy()

        # drop unplottable rows
        need = [x_var, "_Y"] if (x_var in data.columns) else ["_Y"]
        data = data.dropna(subset=[c for c in need if c in data.columns])
        if data.empty:
            return go.Figure().update_layout(title="No plottable points (raw)")

        if not color_col:
            fig = px.scatter(
                data, x=x_var, y="_Y",
                labels={"_Y": y_lab},
                title=f"{y_lab} vs {x_var} (every pitch)"
            )
            fig.update_traces(marker=dict(opacity=0.7, size=8, line=dict(width=0)))
        elif color_var == "GameDate" and "__gameday_num" in data.columns:
            data["__color_date"] = data["GameDate"].dt.strftime("%m/%d/%Y")
            fig = px.scatter(
                data, x=x_var, y="_Y", color="__gameday_num",
                custom_data=["__color_date"],
                labels={"_Y": y_lab, "__gameday_num": "Game Date"},
                title=f"{y_lab} vs {x_var} (every pitch)",
                color_continuous_scale=NONWHITE_SEQ
            )
            vmin, vmax = data["__gameday_num"].min(), data["__gameday_num"].max()
            if pd.notna(vmin) and pd.notna(vmax) and vmin != vmax:
                fig.update_layout(coloraxis_colorbar=dict(
                    title="", tickmode="array",
                    tickvals=[vmin, vmax], ticktext=["older", "recent"]
                ))
            fig.update_traces(
                marker=dict(opacity=0.7, size=8, line=dict(width=0)),
                hovertemplate=f"{x_var}: %{{x}}<br>{y_lab}: %{{y}}<br>"
                              "Game Date: %{customdata[0]}<extra></extra>"
            )
        else:
            # pitch class + count group discrete coloring
            if color_col == "__pitch_class":
                fig = px.scatter(
                    data, x=x_var, y="_Y", color="__pitch_class",
                    color_discrete_map=PITCHCLASS_COLOR_MAP,
                    labels={"_Y": y_lab, "__pitch_class": "Pitch Class"},
                    title=f"{y_lab} vs {x_var} (every pitch)"
                )
            elif color_col == "__count_group":
                fig = px.scatter(
                    data, x=x_var, y="_Y", color="__count_group",
                    color_discrete_map=COUNTGROUP_COLOR_MAP,
                    labels={"_Y": y_lab, "__count_group": "Count"},
                    title=f"{y_lab} vs {x_var} (every pitch)"
                )
            else:
                fig = px.scatter(
                    data, x=x_var, y="_Y", color=color_col,
                    labels={"_Y": y_lab, color_col: color_col},
                    title=f"{y_lab} vs {x_var} (every pitch)"
                )

            fig.update_traces(marker=dict(opacity=0.7, size=8, line=dict(width=0)))

            if color_var == "RunValue":
                lo, hi = _rv_limits(data["RunValue"])
                fig.update_layout(coloraxis_colorbar=dict(title="Run Value"))
                fig.update_coloraxes(colorscale=RV_DIVERGE, cmin=lo, cmid=0, cmax=hi)
            elif color_col and color_col in data.columns and np.issubdtype(data[color_col].dtype, np.number):
                fig.update_coloraxes(colorscale=NONWHITE_SEQ)

    else:
        # AVG mode
        base = df.dropna(subset=[x_var, "_Y"]) if (x_var in df.columns) else df.dropna(subset=["_Y"])
        if base.empty:
            return go.Figure().update_layout(title="No plottable points (avg)")

        if is_cat_color and color_col and color_col in base.columns:
            agg = (
                base.groupby([x_var, color_col], dropna=False)
                    .agg(avg_y=("_Y", "mean"))
                    .reset_index()
            )

            if color_col == "__pitch_class":
                fig = px.scatter(
                    agg, x=x_var, y="avg_y", color="__pitch_class",
                    color_discrete_map=PITCHCLASS_COLOR_MAP,
                    labels={"avg_y": y_lab, "__pitch_class": "Pitch Class"},
                    title=f"{y_lab} vs {x_var} (averages)"
                )
            elif color_col == "__count_group":
                fig = px.scatter(
                    agg, x=x_var, y="avg_y", color="__count_group",
                    color_discrete_map=COUNTGROUP_COLOR_MAP,
                    labels={"avg_y": y_lab, "__count_group": "Count"},
                    title=f"{y_lab} vs {x_var} (averages)"
                )
            else:
                fig = px.scatter(
                    agg, x=x_var, y="avg_y", color=color_col,
                    labels={"avg_y": y_lab, color_col: color_col},
                    title=f"{y_lab} vs {x_var} (averages)"
                )

            fig.update_traces(marker=dict(opacity=0.85, size=9, line=dict(width=0)))
            fig.update_layout(coloraxis=None, coloraxis_showscale=False)

        else:
            if color_var == "GameDate" and "__gameday_num" in base.columns:
                agg = (
                    base.groupby(x_var, dropna=False)
                        .agg(avg_y=("_Y", "mean"), colour_val=("__gameday_num", "mean"))
                        .reset_index()
                )
                agg["__color_date"] = pd.to_datetime(agg["colour_val"] * 86400, unit="s").dt.strftime("%m/%d/%Y")
                fig = px.scatter(
                    agg, x=x_var, y="avg_y", color="colour_val",
                    custom_data=["__color_date"],
                    labels={"avg_y": y_lab, "colour_val": "Game Date"},
                    title=f"{y_lab} vs {x_var} (averages)",
                    color_continuous_scale=NONWHITE_SEQ
                )
                vmin, vmax = agg["colour_val"].min(), agg["colour_val"].max()
                if pd.notna(vmin) and pd.notna(vmax) and vmin != vmax:
                    fig.update_layout(coloraxis_colorbar=dict(
                        title="", tickmode="array",
                        tickvals=[vmin, vmax], ticktext=["older", "recent"]
                    ))
                fig.update_traces(
                    marker=dict(opacity=0.85, size=9, line=dict(width=0)),
                    hovertemplate=f"{x_var}: %{{x}}<br>{y_lab}: %{{y}}<br>"
                                  "Game Date: %{customdata[0]}<extra></extra>"
                )
            else:
                if color_col and color_col in base.columns:
                    base[color_col] = pd.to_numeric(base[color_col], errors="coerce")
                agg = (
                    base.groupby(x_var, dropna=False)
                        .agg(avg_y=("_Y", "mean"), colour_val=(color_col, "mean") if color_col in base.columns else ("_Y","mean"))
                        .reset_index()
                )
                fig = px.scatter(
                    agg, x=x_var, y="avg_y", color="colour_val" if "colour_val" in agg.columns else None,
                    labels={"avg_y": y_lab, "colour_val": (color_var or "")},
                    title=f"{y_lab} vs {x_var} (averages)",
                    color_continuous_scale=NONWHITE_SEQ
                )
                fig.update_traces(marker=dict(opacity=0.85, size=9, line=dict(width=0)))
                if "colour_val" in agg.columns:
                    fig.update_coloraxes(colorscale=NONWHITE_SEQ)

    # global formatting
    fig.update_layout(
        hovermode="closest",
        height=620,
        plot_bgcolor="white",
    )

    # Movement plot formatting (0,0 centered + grids every 5 inches + square axes)
    if x_var == "HORZ_BREAK" and y_var == "INDUCED_VERT_BREAK":
        xvals = pd.to_numeric(df.get("HORZ_BREAK"), errors="coerce").dropna().values
        yvals = pd.to_numeric(df.get("INDUCED_VERT_BREAK"), errors="coerce").dropna().values
        if len(xvals) and len(yvals):
            m = np.nanpercentile(np.abs(np.concatenate([xvals, yvals])), 99)
            r = max(10, int(math.ceil(m / 5.0) * 5))  # round up to next 5
        else:
            r = 20

        fig.update_xaxes(
            range=[-r, r],
            dtick=5,
            showgrid=True,
            zeroline=True,
            zerolinewidth=1,
        )
        fig.update_yaxes(
            range=[-r, r],
            dtick=5,
            showgrid=True,
            zeroline=True,
            zerolinewidth=1,
        )
        fig.update_layout(xaxis=dict(scaleanchor="y", scaleratio=1))

    return fig


# ── 1) Best Get-Me-Over on Hitter’s Counts ──────────────────────
@app.callback(
    Output("gmo-table","data"),
    Output("gmo-table","columns"),
    Output("gmo-table","style_data_conditional"),
    Input("filtered-data","data")
)
def update_gmo_table(sub_json):
    if not sub_json:
        return [], [], []
    df = pd.read_json(sub_json, orient="split")
    HITTER_CT = {"1-0","2-0","2-1","3-0","3-1"}
    data, cols = make_rate_table(df, HITTER_CT, sort_key="CalledPct", desc=True)

# 👉 shrink PitchType to abbreviations and rename header to PT
    for r in data:
        r["PitchType"] = pitch_abbr(r.get("PitchType", ""))
    cols = [{"name": ("PT" if c["id"] == "PitchType" else c["name"]), "id": c["id"]} for c in cols]

    # per-column shading on the five rate columns
    style = per_column_diverging_shading(data, RATE_COLS, alpha=0.25)
    return data, cols, style


# ── 2)  Best Early-Count Strikes  ─────────────────────────────
#      (EARLY counts, any result – strike % like Get-Me-Over)
@app.callback(
    Output("early-strike-table","data"),
    Output("early-strike-table","columns"),
    Output("early-strike-table","style_data_conditional"),
    Input("filtered-data","data"),
)
def update_early_strike_table(sub_json):
    if not sub_json:
        return [], [], []
    df = pd.read_json(sub_json, orient="split")
    EARLY_CT = {f"{b}-{s}" for b in range(4) for s in (0, 1)}
    data, cols = make_rate_table(df, EARLY_CT, sort_key="StrikePct", desc=True)

 # 👉 abbreviate & compact header
    for r in data:
        r["PitchType"] = pitch_abbr(r.get("PitchType", ""))
    cols = [{"name": ("PT" if c["id"] == "PitchType" else c["name"]), "id": c["id"]} for c in cols]

    style = per_column_diverging_shading(data, RATE_COLS, alpha=0.25)
    return data, cols, style



# ── 3) Put-Away on 2-Strikes (Only Swinging & Called) ─────────
@app.callback(
    Output("put2-table","data"),
    Output("put2-table","columns"),
    Output("put2-table","style_data_conditional"),
    Input("filtered-data","data")
)
def update_put2_table(sub_json):
    if not sub_json:
        return [], [], []
    df = pd.read_json(sub_json, orient="split")
    LATE_CT = {"0-2","1-2","2-2","3-2"}
    data, cols = make_rate_table(df, LATE_CT, sort_key="CSW_Pct", desc=True, use_pre=False)


    # 👉 abbreviate & compact header
    for r in data:
        r["PitchType"] = pitch_abbr(r.get("PitchType", ""))
    cols = [{"name": ("PT" if c["id"] == "PitchType" else c["name"]), "id": c["id"]} for c in cols]

    style = per_column_diverging_shading(data, RATE_COLS, alpha=0.25)
    return data, cols, style

# ── 4)  Best Early BIP-Outs  ────────────────────────────────────
@app.callback(
    Output("early-out-table", "data"),
    Output("early-out-table", "columns"),
    Output("early-out-table", "style_data_conditional"),
    Input("filtered-data", "data"),
)
def update_early_bip_table(sub_json):
    if not sub_json:
        return [], [], []

    df = pd.read_json(sub_json, orient="split")

    keep = pitchtype_keep_set(df, MIN_PITCHTYPE_USAGE_TILE, col="PitchType")
    df["_PT"] = _pt_std(df["PitchType"])

    EARLY_CT = {f"{b}-{s}" for b in range(4) for s in (0, 1)}
    df_early = df[df["Count"].isin(EARLY_CT)].copy()
    df_early = df_early[df_early["_PT"].isin(keep)].copy()
    df_early["PitchType"] = df_early["_PT"]


    # Total early attempts by pitch type (denominator for Out%)
    total_by_pt = df_early.groupby("PitchType").size()

    # Only balls in play that became outs
    df_out = df_early[df_early["PitchResult"] == "In Play, Out(s)"].copy()

    # Safety: ensure numerics (in case upstream file had mixed dtypes)
    for c in ("EXIT_VELO", "LAUNCH_ANGLE"):
        if c in df_out.columns:
            df_out[c] = pd.to_numeric(df_out[c], errors="coerce")

    # Summaries
    summary = (
        df_out.groupby("PitchType")
              .agg(
                  Attempts=("PitchResult", "size"),
                  AvgExitVelo=("EXIT_VELO", "mean"),
                  AvgLaunchAng=("LAUNCH_ANGLE", "mean"),
              )
              .reset_index()
    )

    # Out% in early counts for each pitch
    summary["OutPct"] = (
        summary["Attempts"] / summary["PitchType"].map(total_by_pt) * 100
    )

    summary = (
        summary.round({"OutPct": 1, "AvgExitVelo": 1, "AvgLaunchAng": 1})
               .sort_values("OutPct", ascending=False)
    )

    data = summary.to_dict("records")
    cols = [{"name": c, "id": c} for c in summary.columns]

    # Column-wise YlGnBu shading
    style = per_column_diverging_shading(
        data, cols=["OutPct", "AvgExitVelo", "AvgLaunchAng"], alpha=0.25
    )
    return data, cols, style

# ───────────────────────────────────────────────────────────────
# NEW  – Pitch CSW % by # of repetitions within the same PA
# ───────────────────────────────────────────────────────────────
@app.callback(
    Output("occurrence-table", "data"),
    Output("occurrence-table", "columns"),
    Output("occurrence-table", "style_data_conditional"),   # 👈 NEW
    Input("filtered-data", "data"),
)

def update_occurrence_table(sub_json):
    if not sub_json:
        return [], [], []

    df = pd.read_json(sub_json, orient="split")
    data_rows = []
    for _, ab in _group_by_ab(df):
        ab = ab.sort_values("PITCH_NUMBER_AB") if "PITCH_NUMBER_AB" in ab.columns and ab["PITCH_NUMBER_AB"].notna().any() else ab.sort_values("DateTime")
        ab["is_csw"] = ab["PitchResult"].isin(CSW_SET).astype(int)
        # nth occurrence of EACH pitch type inside the AB (1x / 2x / 3x+)
        ab["nth_in_pa"] = ab.groupby("PitchType").cumcount() + 1
        ab["Bucket"]    = ab["nth_in_pa"].apply(lambda n: "1x" if n == 1 else ("2x" if n == 2 else "3x+"))
        data_rows.append(ab[["PitchType","Bucket","is_csw"]])

    if not data_rows:
        return [], [], []

    big = pd.concat(data_rows, ignore_index=True)
    tbl = (big.groupby(["PitchType","Bucket"])["is_csw"].mean().mul(100).round(1)
             .unstack(fill_value=0).reset_index().rename_axis(None, axis=1))
    data = tbl.to_dict("records")
    cols = [{"name": c, "id": c} for c in tbl.columns]
    style = rowwise_ygnbu_shading(data, key_col="PitchType", cols=[c for c in tbl.columns if c in ("1x","2x","3x+")], alpha=0.25)
    return data, cols, style

# ───────────────────────────────────────────────────────────────
# LOCATION PLOT CALLBACKS
# ───────────────────────────────────────────────────────────────

#1
@app.callback(
    Output("loc-table", "data"),
    Output("loc-table", "columns"),
    Input("filtered-data", "data"),
    Input("loc-selected-ids", "data"),
)
def loc_update_table(sub_json, selected_ids):
    if not sub_json:
        return [], []
    df = pd.read_json(sub_json, orient="split")

    # Ensure row id exists
    if "_row_id" not in df.columns:
        df = df.reset_index(drop=True)
        df["_row_id"] = df.index.astype(int)

    # Apply lasso selection (if any)
    if selected_ids:
        df = df[df["_row_id"].isin(selected_ids)]

    # Keep table sane (optional but recommended)
    preferred = [
        "GameDate","Pitcher","BatterHand","Count","PitchType","PitchResult","Velo",
        "plate_loc_side","plate_loc_height","RunValue"
    ]
    show = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    df = df[show]

    columns = [{"name": c, "id": c} for c in df.columns]
    return df.to_dict("records"), columns

#2
@app.callback(
    Output("loc-summary-table", "columns"),
    Output("loc-summary-table", "data"),
    Output("loc-zone-metric", "children"),
    Input("loc-table", "derived_virtual_data"),
    State("loc-table", "data"),
)
def loc_update_summary(derived, rows):
    df_view = pd.DataFrame(derived) if derived is not None else pd.DataFrame(rows or [])
    if df_view.empty:
        return [], [], ""

    df_view = _ensure_metrics(df_view)  # from LocationPlots :contentReference[oaicite:13]{index=13}
    cols, data = make_summary_only(df_view)  # from LocationPlots :contentReference[oaicite:14]{index=14}

    zone_pct = float(df_view["InZone"].mean() * 100.0) if len(df_view) else 0.0
    return cols, data, f"Zone%: {zone_pct:.1f}"

#3
@app.callback(
    Output("loc-plate-graph", "figure"),
    Input("loc-table", "derived_virtual_data"),
    State("loc-table", "data"),
    Input("loc-color-by", "value"),
    Input("loc-plot-type", "value"),
    Input("loc-flags", "value"),
)

def loc_update_plot(derived, rows, color_by, plot_type, flags):
    df_view = pd.DataFrame(derived) if derived is not None else pd.DataFrame(rows or [])
    if df_view.empty:
        return go.Figure().update_layout(title="Plate Location")

    # Ensure numeric plate coords (protects against dtype=str ingestion)
    for c in ("plate_loc_side", "plate_loc_height"):
        if c in df_view.columns:
            df_view[c] = pd.to_numeric(df_view[c], errors="coerce")

    # If coords missing, nothing to plot
    if not {"plate_loc_side", "plate_loc_height"} <= set(df_view.columns):
        return go.Figure().update_layout(title="Plate Location (missing plate_loc cols)")

    df_view = df_view.dropna(subset=["plate_loc_side", "plate_loc_height"])
    if df_view.empty:
        return go.Figure().update_layout(title="Plate Location (no valid locations after filters)")

    catcher_view = "flip" in (flags or [])
    show_zone = "zone" in (flags or [])

    if plot_type == "heatmap":
        return build_heatmap(df_view, catcher_view, show_zone)

    # ✅ correct argument order
    return build_scatter(df_view, color_by, catcher_view, show_zone)



#4
@app.callback(
    Output("loc-selected-ids", "data"),
    Input("loc-plate-graph", "selectedData"),
    Input("loc-clear-selection", "n_clicks"),
    Input("loc-plot-type", "value"),
    State("loc-selected-ids", "data"),
    prevent_initial_call=True,
)
def loc_capture_selection(selectedData, n_clear, plot_type, prev):
    ctx = dash.callback_context
    if not ctx.triggered:
        return dash.no_update
    trig = ctx.triggered[0]["prop_id"].split(".")[0]

    # switching to heatmap or clicking clear nukes selection
    if trig in ("loc-clear-selection", "loc-plot-type"):
        return []

    if not selectedData or "points" not in selectedData:
        return []

    ids = []
    for p in selectedData["points"]:
        cd = p.get("customdata")
        if cd is None:
            continue
        # custom_data=["_row_id"] => cd is like [row_id]
        rid = cd[0] if isinstance(cd, (list, tuple)) else cd
        ids.append(int(rid))
    return sorted(set(ids))

#5
@app.callback(
    Output("loc-color-by", "options"),
    Input("filtered-data", "data"),
)
def loc_set_colorby_options(sub_json):
    if not sub_json:
        return []
    df = pd.read_json(sub_json, orient="split")
    cols = [c for c in df.columns if c not in {"_row_id"}]
    return [{"label": c, "value": c} for c in cols]


# 7) RUN (Colab inline) ─────────────────────────────────────────────

# new
app.run(jupyter_mode="inline", jupyter_height=1400, jupyter_width="100%", debug=False)



<IPython.core.display.Javascript object>